In [1]:
!pip install torch --quiet

# Setup

In [2]:
# ─────────────────────────────────────────────
# CELL 1 — Imports
# ─────────────────────────────────────────────
import os
import json
import random
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# ─────────────────────────────────────────────
# CELL 2 — Paths & Output Directories
# ─────────────────────────────────────────────

# ── Inputs ───────────────────────────────────
DATA_DIR       = Path("data/pp5_suvr")                          # preprocessed PET images
CSV_PATH       = Path("amy_dataset_final.csv")             # labels + demographics
SPLITS_PATH    = Path("cv_splits/amyloid_cv_splits.json")  # 7-fold CV splits
HOLDOUT_PATH   = Path("cv_splits/holdout_inference_set.json")

# ── Outputs ──────────────────────────────────
DIR_BEST       = Path("simpleCNN/simpleCNN_best_config")      # best arch + hparams from search
DIR_CKPT       = Path("simpleCNN/simpleCNN_checkpoints")      # per-fold final weights
DIR_RESULTS    = Path("simpleCNN/simpleCNN_results")          # summaries, plots, CSVs

for d in [DIR_BEST, DIR_CKPT, DIR_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Global training constants ─────────────────
IMAGE_SHAPE    = (91, 109, 91)   # D x H x W  (2mm MNI)
N_FOLDS        = 7
SEARCH_FOLD    = 0               # proxy fold for random search
N_TRIALS       = 50
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Output dirs ready:")
print(f"  Best config  → {DIR_BEST}")
print(f"  Checkpoints  → {DIR_CKPT}")
print(f"  Results      → {DIR_RESULTS}")
print(f"Device: {DEVICE}")

Output dirs ready:
  Best config  → simpleCNN/simpleCNN_best_config
  Checkpoints  → simpleCNN/simpleCNN_checkpoints
  Results      → simpleCNN/simpleCNN_results
Device: cuda


# Dataset Class

In [4]:
# ─────────────────────────────────────────────
# CELL 3 — Dataset Class
# ─────────────────────────────────────────────

class AmyloidDataset(Dataset):
    """
    Loads preprocessed amyloid PET images (.nii.gz) and returns
    (image_tensor, label) pairs.

    Augmentation (training only):
      - Random left-right flip (axis 0)        p=0.5
      - Random up-down flip (axis 1)           p=0.5
      - Random small intensity jitter          ±2% of image mean
    """

    def __init__(self, records: list[dict], data_dir: Path, augment: bool = False):
        """
        records  : list of dicts with keys 'image_id_2_str' and 'AMYLOID_STATUS'
        data_dir : path to pp5_suvr/
        augment  : True for train splits, False for val/test/holdout
        """
        self.records  = records
        self.data_dir = data_dir
        self.augment  = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        label = int(rec["AMYLOID_STATUS"])

        # ── Load image ───────────────────────
        fpath = self.data_dir / f"{rec['image_id_2_str']}.nii.gz"
        img   = nib.load(str(fpath)).get_fdata(dtype=np.float32)  # (91,109,91)

        # ── Augmentation (train only) ─────────
        if self.augment:
            if random.random() < 0.5:
                img = np.flip(img, axis=0).copy()   # L-R flip
            if random.random() < 0.5:
                img = np.flip(img, axis=1).copy()   # U-D flip
            jitter = (random.random() * 0.04 - 0.02) * img.mean()
            img = img + jitter

        # ── To tensor: add channel dim → (1, D, H, W) ────
        img = torch.from_numpy(img).unsqueeze(0)   # (1, 91, 109, 91)

        return img, torch.tensor(label, dtype=torch.long)


def make_records(df: pd.DataFrame, patient_list: list) -> list[dict]:
    """Filter dataframe to patients in list, return list of dicts."""
    subset = df[df["PTID"].isin(patient_list)]
    return subset[["image_id_2_str", "AMYLOID_STATUS", "PTID"]].to_dict("records")


# ── Quick smoke test ──────────────────────────
df_test = pd.read_csv(CSV_PATH)
with open(SPLITS_PATH) as f:
    splits_test = json.load(f)

fold0      = splits_test[SEARCH_FOLD]
train_recs = make_records(df_test, fold0["train_patients"])
ds_train   = AmyloidDataset(train_recs, DATA_DIR, augment=True)
img, lbl   = ds_train[0]

print(f"Image shape   : {img.shape}")    # expect torch.Size([1, 91, 109, 91])
print(f"Label         : {lbl.item()}")
print(f"Dtype         : {img.dtype}")
print(f"Min / Max     : {img.min():.4f} / {img.max():.4f}")
print(f"Train set size: {len(ds_train)}")

Image shape   : torch.Size([1, 91, 109, 91])
Label         : 1
Dtype         : torch.float32
Min / Max     : -0.0034 / 4.5541
Train set size: 1005


# Helper Functions

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Helper Functions
# ─────────────────────────────────────────────

# ── 1. Focal Loss ─────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal loss for binary classification.
    Down-weights easy examples, focuses on hard ones.
    gamma=0 reduces to standard cross-entropy.
    """
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce   = F.cross_entropy(logits, targets, reduction="none")
        probs = torch.exp(-ce)
        return ((1 - probs) ** self.gamma * ce).mean()


# ── 2. Threshold Tuning ───────────────────────
def tune_threshold(labels: np.ndarray, probs: np.ndarray) -> float:
    """
    Sweep thresholds [0.1 → 0.9] on validation set.
    Returns threshold that maximises macro-F1.
    """
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.4, 0.61, 0.01):
        preds = (probs >= t).astype(int)
        f1    = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


# ── 3. Single-epoch train / eval passes ───────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        n           += len(labels)
    return total_loss / n


@torch.no_grad()
def eval_pass(model, loader, criterion, device):
    """Returns (avg_loss, labels_np, probs_np)."""
    model.eval()
    total_loss, n = 0.0, 0
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        n           += len(labels)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs)
    labels_np = np.concatenate(all_labels)
    probs_np  = np.concatenate(all_probs)
    return total_loss / n, labels_np, probs_np


# ── 4. Full Evaluation (after threshold tuning) ──
def evaluate(labels: np.ndarray, probs: np.ndarray, threshold: float) -> dict:
    """
    Returns a dict with AUC, macro-F1, per-class F1,
    sensitivity, specificity, and confusion matrix.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    report = classification_report(labels, preds,
                                   target_names=["Amyloid-", "Amyloid+"],
                                   output_dict=True, zero_division=0)
    return {
        "auc"          : roc_auc_score(labels, probs),
        "macro_f1"     : f1_score(labels, preds, average="macro", zero_division=0),

        "f1_neg"       : report["Amyloid-"]["f1-score"],
        "precision_neg": report["Amyloid-"]["precision"],
        "recall_neg"   : report["Amyloid-"]["recall"],

        "f1_pos"       : report["Amyloid+"]["f1-score"],
        "precision_pos": report["Amyloid+"]["precision"],
        "recall_pos"   : report["Amyloid+"]["recall"],

        "sensitivity"  : tp / (tp + fn) if (tp + fn) > 0 else 0.0,  # = recall_pos
        "specificity"  : tn / (tn + fp) if (tn + fp) > 0 else 0.0,  # = recall_neg
        "threshold"    : threshold,
        "confusion"    : [[int(tn), int(fp)], [int(fn), int(tp)]],
    }


# ── 5. Plot: Train vs Val Loss Curve ──────────
def plot_loss_curve(train_losses: list, val_losses: list,
                    title: str, save_path: Path):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label="Train loss", linewidth=1.8)
    ax.plot(val_losses,   label="Val loss",   linewidth=1.8, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Focal Loss")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# ── 6. Plot: ROC Curve ────────────────────────
def plot_roc(labels: np.ndarray, probs: np.ndarray,
             title: str, save_path: Path):
    fpr, tpr, _ = roc_curve(labels, probs)
    auc         = roc_auc_score(labels, probs)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f"AUC = {auc:.3f}", linewidth=2)
    ax.plot([0,1],[0,1], "k--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# ── 7. Plot: Confusion Matrix ─────────────────
def plot_confusion(cm: list, title: str, save_path: Path):
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(np.array(cm), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred -", "Pred +"],
                yticklabels=["True -", "True +"], ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


print("Helper functions ready ✓")

# SimpleCNN

In [6]:
# ─────────────────────────────────────────────
# CELL 5 — Model Architecture
# ─────────────────────────────────────────────

class Simple3DCNN(nn.Module):
    """
    Configurable 3D CNN for amyloid PET binary classification.

    Architecture:
        N convolutional blocks  →  Global Average Pooling  →  Classifier head
    
    Each conv block:
        Conv3d → BatchNorm3d → ReLU → MaxPool3d(2)

    Config parameters (all tunable in random search):
        channels      : tuple of ints, one per block  e.g. (32, 64, 128)
        fc_units      : number of units in the FC layer
        dropout       : dropout rate before FC
        kernel_size   : conv kernel size (same for all blocks)
    """

    def __init__(self,
                 channels   : tuple = (32, 64, 128),
                 fc_units   : int   = 256,
                 dropout    : float = 0.5,
                 kernel_size: int   = 3):
        super().__init__()

        blocks = []
        in_ch  = 1                          # single-channel PET image
        for out_ch in channels:
            blocks += [
                nn.Conv3d(in_ch, out_ch, kernel_size=kernel_size,
                          padding=kernel_size // 2, bias=False),
                nn.BatchNorm3d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool3d(kernel_size=2, stride=2),
            ]
            in_ch = out_ch

        self.encoder = nn.Sequential(*blocks)
        self.gap      = nn.AdaptiveAvgPool3d(1)   # → (B, C, 1, 1, 1)
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(in_ch, fc_units)
        self.head     = nn.Linear(fc_units, 2)    # binary: neg / pos

    def forward(self, x):
        x = self.encoder(x)           # (B, C, D', H', W')
        x = self.gap(x).flatten(1)    # (B, C)
        x = self.dropout(x)
        x = F.relu(self.fc(x))
        return self.head(x)           # (B, 2) — raw logits


# ── Smoke test ────────────────────────────────
dummy_input = torch.zeros(2, 1, 91, 109, 91)   # batch of 2

for channels in [(32,), (32, 64), (32, 64, 128), (32, 64, 128, 256)]:
    model  = Simple3DCNN(channels=channels)
    out    = model(dummy_input)
    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  channels={str(channels):<22} output={out.shape}  params={nparams:,}")

print("\nModel architecture ready ✓")

  channels=(32,)                  output=torch.Size([2, 2])  params=9,890
  channels=(32, 64)               output=torch.Size([2, 2])  params=73,506
  channels=(32, 64, 128)          output=torch.Size([2, 2])  params=311,330
  channels=(32, 64, 128, 256)     output=torch.Size([2, 2])  params=1,229,346

Model architecture ready ✓


## Random search

In [7]:
# ─────────────────────────────────────────────
# CELL 6 — Random Search (Fold 0 proxy)
# ─────────────────────────────────────────────

SEARCH_EPOCHS_MAX = 25
PATIENCE          = 10
N_TRIALS          = 20

# ── Search space ──────────────────────────────
SEARCH_SPACE = {
    "channels"    : [
        (32,),
        (64,),
        (32, 64),
        (64, 128),
        (32, 64, 128),
        (64, 128, 256),
        (32, 64, 128, 256),
        (64, 128, 256, 512),
    ],
    "fc_units"    : [128, 256, 512],
    "dropout"     : [0.3, 0.4, 0.5, 0.6],
    "kernel_size" : [3, 5],
    "lr"          : [1e-4, 3e-4, 5e-4, 1e-3],
    "batch_size"  : [4, 8, 16],
    "gamma"       : [0.5, 1.0, 2.0, 3.0],
}


def sample_config(space: dict, seed: int) -> dict:
    rng = random.Random(seed)
    return {k: rng.choice(v) for k, v in space.items()}


def run_trial(config: dict, df: pd.DataFrame,
              splits: list, fold_idx: int,
              trial_idx: int) -> dict:

    print(f"    Loading data for fold {fold_idx}...")
    fold       = splits[fold_idx]
    train_recs = make_records(df, fold["train_patients"])
    val_recs   = make_records(df, fold["val_patients"])
    print(f"    Train: {len(train_recs)} images  |  Val: {len(val_recs)} images")

    train_loader = DataLoader(
        AmyloidDataset(train_recs, DATA_DIR, augment=True),
        batch_size=config["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        AmyloidDataset(val_recs, DATA_DIR, augment=False),
        batch_size=config["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )

    model = Simple3DCNN(
                channels    = config["channels"],
                fc_units    = config["fc_units"],
                dropout     = config["dropout"],
                kernel_size = config["kernel_size"],
            ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Model: {config['channels']}  fc={config['fc_units']}  "
          f"drop={config['dropout']}  k={config['kernel_size']}  "
          f"params={n_params:,}")
    print(f"    Optim: lr={config['lr']}  bs={config['batch_size']}  "
          f"gamma={config['gamma']}  max_epochs={SEARCH_EPOCHS_MAX}")

    criterion = FocalLoss(gamma=config["gamma"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"],
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=SEARCH_EPOCHS_MAX)

    best_val_f1    = -1.0
    best_threshold = 0.5
    best_val_labels = None
    best_val_probs  = None
    patience_count  = 0
    train_losses, val_losses = [], []

    print(f"    Training...")
    for epoch in range(SEARCH_EPOCHS_MAX):
        tr_loss          = train_one_epoch(model, train_loader,
                                           optimizer, criterion, DEVICE)
        val_loss, vl, vp = eval_pass(model, val_loader, criterion, DEVICE)
        scheduler.step()

        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        t      = tune_threshold(vl, vp)
        val_f1 = f1_score(vl, (vp >= t).astype(int),
                          average="macro", zero_division=0)
        val_auc = roc_auc_score(vl, vp)

        # print every epoch
        print(f"      ep {epoch+1:02d}/{SEARCH_EPOCHS_MAX} | "
              f"train_loss={tr_loss:.4f}  val_loss={val_loss:.4f} | "
              f"val_f1={val_f1:.4f}  val_auc={val_auc:.4f}  "
              f"thresh={t:.2f}  "
              f"patience={patience_count}/{PATIENCE}"
              + (" ← best" if val_f1 > best_val_f1 else ""))

        if val_f1 > best_val_f1:
            best_val_f1     = val_f1
            best_threshold  = t
            best_val_labels = vl
            best_val_probs  = vp
            patience_count  = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"    Early stopping at epoch {epoch+1} "
                      f"(patience={PATIENCE} exhausted)")
                break

    metrics = evaluate(best_val_labels, best_val_probs, best_threshold)

    print(f"    ── Trial {trial_idx+1} result ──────────────────────")
    print(f"       val_macro_f1 : {metrics['macro_f1']:.4f}")
    print(f"       val_auc      : {metrics['auc']:.4f}")
    print(f"       f1_neg       : {metrics['f1_neg']:.4f}  "
          f"(prec={metrics['precision_neg']:.4f}  rec={metrics['recall_neg']:.4f})")
    print(f"       f1_pos       : {metrics['f1_pos']:.4f}  "
          f"(prec={metrics['precision_pos']:.4f}  rec={metrics['recall_pos']:.4f})")
    print(f"       threshold    : {best_threshold:.2f}  |  "
          f"epochs_run: {len(train_losses)}")

    return {
        "trial"            : trial_idx,
        "config"           : config,
        "val_macro_f1"     : metrics["macro_f1"],
        "val_auc"          : metrics["auc"],
        "val_f1_neg"       : metrics["f1_neg"],
        "val_f1_pos"       : metrics["f1_pos"],
        "val_precision_neg": metrics["precision_neg"],
        "val_recall_neg"   : metrics["recall_neg"],
        "val_precision_pos": metrics["precision_pos"],
        "val_recall_pos"   : metrics["recall_pos"],
        "best_threshold"   : best_threshold,
        "epochs_run"       : len(train_losses),
        "train_losses"     : train_losses,
        "val_losses"       : val_losses,
    }

In [8]:
# ── Run the search ────────────────────────────
df = pd.read_csv(CSV_PATH)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

all_results  = []
search_start = time.time()

print(f"{'='*60}")
print(f"  Random Search — {N_TRIALS} trials on fold {SEARCH_FOLD}")
print(f"  max_epochs={SEARCH_EPOCHS_MAX}  patience={PATIENCE}")
print(f"{'='*60}\n")

for trial in range(N_TRIALS):
    config       = sample_config(SEARCH_SPACE, seed=SEED + trial)
    trial_start  = time.time()

    print(f"{'─'*60}")
    print(f"  TRIAL {trial+1:02d}/{N_TRIALS}")
    print(f"{'─'*60}")

    try:
        result = run_trial(config, df, splits, SEARCH_FOLD, trial)
        all_results.append(result)

        # running best so far
        best_so_far = max(all_results, key=lambda x: x["val_macro_f1"])
        trial_time  = time.time() - trial_start
        elapsed     = time.time() - search_start
        remaining   = (elapsed / (trial + 1)) * (N_TRIALS - trial - 1)

        print(f"    Trial time : {trial_time/60:.1f} min")
        print(f"    Elapsed    : {elapsed/3600:.2f}h  |  "
              f"ETA: {remaining/3600:.2f}h")
        print(f"    Best so far: trial {best_so_far['trial']+1}  "
              f"macro_f1={best_so_far['val_macro_f1']:.4f}  "
              f"auc={best_so_far['val_auc']:.4f}  "
              f"config={best_so_far['config']['channels']}")

    except Exception as e:
        print(f"    FAILED — {e}")
        import traceback; traceback.print_exc()

    # save after every trial (safe against crashes)
    results_path = DIR_BEST / "search_results.json"
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"    Saved {len(all_results)} trials → {results_path}\n")

  Random Search — 20 trials on fold 0
  max_epochs=25  patience=10

────────────────────────────────────────────────────────────
  TRIAL 01/20
────────────────────────────────────────────────────────────
    Loading data for fold 0...
    Train: 1005 images  |  Val: 141 images
    Model: (64,)  fc=128  drop=0.5  k=3  params=10,434
    Optim: lr=0.0003  bs=4  gamma=0.5  max_epochs=25
    Training...
      ep 01/25 | train_loss=0.4904  val_loss=0.4835 | val_f1=0.6871  val_auc=0.7141  thresh=0.47  patience=0/10 ← best
      ep 02/25 | train_loss=0.4912  val_loss=0.4889 | val_f1=0.5053  val_auc=0.7200  thresh=0.51  patience=0/10
      ep 03/25 | train_loss=0.4907  val_loss=0.4876 | val_f1=0.5870  val_auc=0.6775  thresh=0.50  patience=1/10
      ep 04/25 | train_loss=0.4925  val_loss=0.4835 | val_f1=0.5553  val_auc=0.6400  thresh=0.44  patience=2/10
      ep 05/25 | train_loss=0.4919  val_loss=0.4859 | val_f1=0.5289  val_auc=0.6840  thresh=0.49  patience=3/10
      ep 06/25 | train_loss=0.4

Traceback (most recent call last):
  File "/tmp/ipykernel_114187/1389392474.py", line 23, in <module>
    result = run_trial(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1936029980.py", line 85, in run_trial
    tr_loss          = train_one_epoch(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/3591577170.py", line 44, in train_one_epoch
    loss = criterion(model(imgs), labels)
                     ^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1778, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1789, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1844773771.py", line 48

    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 4 trials → simpleCNN/simpleCNN_best_config/search_results.json

────────────────────────────────────────────────────────────
  TRIAL 07/20
────────────────────────────────────────────────────────────
    Loading data for fold 0...
    Train: 1005 images  |  Val: 141 images
    Model: (64, 128, 256)  fc=128  drop=0.5  k=3  params=1,141,698
    Optim: lr=0.001  bs=4  gamma=1.0  max_epochs=25
    Training...


Traceback (most recent call last):
  File "/tmp/ipykernel_114187/1389392474.py", line 23, in <module>
    result = run_trial(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1936029980.py", line 85, in run_trial
    tr_loss          = train_one_epoch(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/3591577170.py", line 45, in train_one_epoch
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^

      ep 01/25 | train_loss=0.3546  val_loss=0.3384 | val_f1=0.6591  val_auc=0.6825  thresh=0.54  patience=0/10 ← best
      ep 02/25 | train_loss=0.3400  val_loss=0.3214 | val_f1=0.7020  val_auc=0.7629  thresh=0.44  patience=0/10 ← best
      ep 03/25 | train_loss=0.3256  val_loss=0.3187 | val_f1=0.7659  val_auc=0.8775  thresh=0.60  patience=0/10 ← best
      ep 04/25 | train_loss=0.3037  val_loss=0.3558 | val_f1=0.6453  val_auc=0.8818  thresh=0.60  patience=0/10
      ep 05/25 | train_loss=0.2943  val_loss=0.2423 | val_f1=0.8899  val_auc=0.9469  thresh=0.46  patience=1/10 ← best
      ep 06/25 | train_loss=0.2892  val_loss=0.2381 | val_f1=0.8667  val_auc=0.8781  thresh=0.60  patience=0/10
      ep 07/25 | train_loss=0.2767  val_loss=0.2546 | val_f1=0.8499  val_auc=0.8938  thresh=0.40  patience=1/10
      ep 08/25 | train_loss=0.2690  val_loss=0.4544 | val_f1=0.3562  val_auc=0.9371  thresh=0.40  patience=2/10
      ep 09/25 | train_loss=0.2666  val_loss=0.2223 | val_f1=0.8822  val_auc

...

Traceback (most recent call last):
  File "/tmp/ipykernel_114187/1389392474.py", line 23, in <module>
    result = run_trial(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1936029980.py", line 85, in run_trial
    tr_loss          = train_one_epoch(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/3591577170.py", line 45, in train_one_epoch
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^

    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 9 trials → simpleCNN/simpleCNN_best_config/search_results.json

────────────────────────────────────────────────────────────
  TRIAL 19/20
────────────────────────────────────────────────────────────
    Loading data for fold 0...
    Train: 1005 images  |  Val: 141 images
    Model: (32, 64, 128)  fc=256  drop=0.4  k=5  params=1,317,986
    Optim: lr=0.0003  bs=8  gamma=3.0  max_epochs=25
    Training...


Traceback (most recent call last):
  File "/tmp/ipykernel_114187/1389392474.py", line 23, in <module>
    result = run_trial(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1936029980.py", line 85, in run_trial
    tr_loss          = train_one_epoch(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/3591577170.py", line 45, in train_one_epoch
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^

      ep 01/25 | train_loss=0.0866  val_loss=0.0840 | val_f1=0.5957  val_auc=0.6496  thresh=0.50  patience=0/10 ← best
      ep 02/25 | train_loss=0.0850  val_loss=0.1007 | val_f1=0.7791  val_auc=0.8474  thresh=0.60  patience=0/10 ← best
      ep 03/25 | train_loss=0.0779  val_loss=0.0777 | val_f1=0.8002  val_auc=0.8641  thresh=0.42  patience=0/10 ← best
      ep 04/25 | train_loss=0.0713  val_loss=0.3106 | val_f1=0.3230  val_auc=0.9385  thresh=0.45  patience=0/10
      ep 05/25 | train_loss=0.0763  val_loss=0.0759 | val_f1=0.7497  val_auc=0.8274  thresh=0.48  patience=1/10
      ep 06/25 | train_loss=0.0720  val_loss=0.1174 | val_f1=0.7425  val_auc=0.8993  thresh=0.60  patience=2/10
      ep 07/25 | train_loss=0.0643  val_loss=0.2264 | val_f1=0.3562  val_auc=0.8877  thresh=0.40  patience=3/10
      ep 08/25 | train_loss=0.0594  val_loss=0.1065 | val_f1=0.8297  val_auc=0.9430  thresh=0.60  patience=4/10 ← best
      ep 09/25 | train_loss=0.0587  val_loss=0.1738 | val_f1=0.6098  val_auc

Traceback (most recent call last):
  File "/tmp/ipykernel_114187/1389392474.py", line 23, in <module>
    result = run_trial(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1936029980.py", line 85, in run_trial
    tr_loss          = train_one_epoch(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/3591577170.py", line 45, in train_one_epoch
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^

In [9]:
with open(DIR_BEST / "search_results.json") as f:
    saved = json.load(f)

print(f"Trials saved: {len(saved)}\n")
for r in saved:
    nparams = sum(p.numel() for p in Simple3DCNN(
        channels    = tuple(r["config"]["channels"]),
        fc_units    = r["config"]["fc_units"],
        dropout     = r["config"]["dropout"],
        kernel_size = r["config"]["kernel_size"],
    ).parameters() if p.requires_grad)
    print(f"  Trial {r['trial']+1:02d} | macro_f1={r['val_macro_f1']:.4f} | "
          f"auc={r['val_auc']:.4f} | params={nparams:>12,} | "
          f"config={r['config']['channels']}")

Trials saved: 10

  Trial 01 | macro_f1=0.6871 | auc=0.7141 | params=      10,434 | config=[64]
  Trial 02 | macro_f1=0.6767 | auc=0.7249 | params=      13,026 | config=[32]
  Trial 03 | macro_f1=0.9355 | auc=0.9748 | params=   1,295,650 | config=[32, 64, 128, 256]
  Trial 04 | macro_f1=0.9136 | auc=0.9389 | params=   1,317,986 | config=[32, 64, 128]
  Trial 07 | macro_f1=0.9063 | auc=0.9565 | params=   1,141,698 | config=[64, 128, 256]
  Trial 11 | macro_f1=0.9051 | auc=0.9503 | params=   1,301,218 | config=[32, 64, 128]
  Trial 13 | macro_f1=0.7198 | auc=0.7548 | params=     277,346 | config=[32, 64]
  Trial 15 | macro_f1=0.5812 | auc=0.7599 | params=      13,026 | config=[32]
  Trial 16 | macro_f1=0.6611 | auc=0.7045 | params=       9,890 | config=[32]
  Trial 19 | macro_f1=0.8996 | auc=0.9501 | params=   1,317,986 | config=[32, 64, 128]


In [10]:
sorted_results = sorted(saved, key=lambda x: x["val_macro_f1"], reverse=True)

print(f"{'Rank':<6} {'Trial':<8} {'macro_f1':<12} {'AUC':<10} {'params':>12}  config")
print("─" * 65)
for i, r in enumerate(sorted_results[:5]):
    nparams = sum(p.numel() for p in Simple3DCNN(
        channels    = tuple(r["config"]["channels"]),
        fc_units    = r["config"]["fc_units"],
        dropout     = r["config"]["dropout"],
        kernel_size = r["config"]["kernel_size"],
    ).parameters() if p.requires_grad)
    print(f"  {i+1:<4} {r['trial']+1:<8} {r['val_macro_f1']:<12.4f} "
          f"{r['val_auc']:<10.4f} {nparams:>12,}  {r['config']['channels']}")

Rank   Trial    macro_f1     AUC              params  config
─────────────────────────────────────────────────────────────────
  1    3        0.9355       0.9748        1,295,650  [32, 64, 128, 256]
  2    4        0.9136       0.9389        1,317,986  [32, 64, 128]
  3    7        0.9063       0.9565        1,141,698  [64, 128, 256]
  4    11       0.9051       0.9503        1,301,218  [32, 64, 128]
  5    19       0.8996       0.9501        1,317,986  [32, 64, 128]


In [11]:
# ── Automatic selection — best macro_f1 with params penalty ──
PARAM_LIMIT = 5_000_000   # reject models above 5M params

candidates = []
for r in saved:
    nparams = sum(p.numel() for p in Simple3DCNN(
        channels    = tuple(r["config"]["channels"]),
        fc_units    = r["config"]["fc_units"],
        dropout     = r["config"]["dropout"],
        kernel_size = r["config"]["kernel_size"],
    ).parameters() if p.requires_grad)
    if nparams <= PARAM_LIMIT:
        candidates.append((r, nparams))

# rank by macro_f1
candidates.sort(key=lambda x: x[0]["val_macro_f1"], reverse=True)
chosen_result, chosen_params = candidates[0]
CHOSEN_TRIAL = chosen_result["trial"] + 1

print(f"  Param limit   : {PARAM_LIMIT:,}")
print(f"  Candidates    : {len(candidates)}/{len(saved)} trials under limit")
print(f"  Auto-selected : Trial {CHOSEN_TRIAL}  "
      f"macro_f1={chosen_result['val_macro_f1']:.4f}  "
      f"params={chosen_params:,}")
# save to disk
config_path = DIR_BEST / "best_config.json"
with open(config_path, "w") as f:
    json.dump(chosen_result, f, indent=2, default=str)

# print summary
print(f"\n{'='*60}")
print(f"  Selected config — Trial {CHOSEN_TRIAL}")
print(f"{'='*60}")
print(f"  Macro F1   : {chosen_result['val_macro_f1']:.4f}")
print(f"  AUC        : {chosen_result['val_auc']:.4f}")
print(f"  f1_neg     : {chosen_result['val_f1_neg']:.4f}  "
      f"(prec={chosen_result['val_precision_neg']:.4f}  rec={chosen_result['val_recall_neg']:.4f})")
print(f"  f1_pos     : {chosen_result['val_f1_pos']:.4f}  "
      f"(prec={chosen_result['val_precision_pos']:.4f}  rec={chosen_result['val_recall_pos']:.4f})")
print(f"  Threshold  : {chosen_result['best_threshold']:.2f}")
print(f"  Epochs run : {chosen_result['epochs_run']}")
print(f"{'─'*60}")
print(f"  channels   : {chosen_result['config']['channels']}")
print(f"  fc_units   : {chosen_result['config']['fc_units']}")
print(f"  dropout    : {chosen_result['config']['dropout']}")
print(f"  kernel_size: {chosen_result['config']['kernel_size']}")
print(f"  lr         : {chosen_result['config']['lr']}")
print(f"  batch_size : {chosen_result['config']['batch_size']}")
print(f"  gamma      : {chosen_result['config']['gamma']}")
print(f"{'='*60}")
print(f"  Saved → {config_path}")

# also save the loss curve for the chosen trial
plot_loss_curve(
    chosen_result["train_losses"],
    chosen_result["val_losses"],
    title=f"Search — Trial {CHOSEN_TRIAL} Loss Curve (chosen config)",
    save_path=DIR_BEST / "chosen_trial_loss_curve.png"
)
print(f"  Loss curve → {DIR_BEST / 'chosen_trial_loss_curve.png'}")

  Param limit   : 5,000,000
  Candidates    : 10/10 trials under limit
  Auto-selected : Trial 3  macro_f1=0.9355  params=1,295,650

  Selected config — Trial 3
  Macro F1   : 0.9355
  AUC        : 0.9748
  f1_neg     : 0.9419  (prec=0.9481  rec=0.9359)
  f1_pos     : 0.9291  (prec=0.9219  rec=0.9365)
  Threshold  : 0.44
  Epochs run : 25
────────────────────────────────────────────────────────────
  channels   : [32, 64, 128, 256]
  fc_units   : 512
  dropout    : 0.3
  kernel_size: 3
  lr         : 0.001
  batch_size : 4
  gamma      : 2.0
  Saved → simpleCNN/simpleCNN_best_config/best_config.json
  Loss curve → simpleCNN/simpleCNN_best_config/chosen_trial_loss_curve.png


## Full CV Training

In [12]:
# ─────────────────────────────────────────────
# CELL 7 — Full 7-Fold CV Training
# ─────────────────────────────────────────────

# ── Load best config ──────────────────────────
with open(DIR_BEST / "best_config.json") as f:
    best = json.load(f)

CFG = best["config"]
print("Loaded config:")
for k, v in CFG.items():
    print(f"  {k:12s}: {v}")

# ── CV training constants ─────────────────────
CV_EPOCHS   = 50
CV_PATIENCE = 15

print(f"\nCV settings: {CV_EPOCHS} max epochs, patience={CV_PATIENCE}")
print(f"Folds: {N_FOLDS}")

Loaded config:
  channels    : [32, 64, 128, 256]
  fc_units    : 512
  dropout     : 0.3
  kernel_size : 3
  lr          : 0.001
  batch_size  : 4
  gamma       : 2.0

CV settings: 50 max epochs, patience=15
Folds: 7


In [13]:
def train_fold(fold_idx: int, df: pd.DataFrame,
               splits: list, cfg: dict) -> dict:

    fold       = splits[fold_idx]
    train_recs = make_records(df, fold["train_patients"])
    val_recs   = make_records(df, fold["val_patients"])
    test_recs  = make_records(df, fold["test_patients"])

    print(f"\n  Train: {len(train_recs)}  Val: {len(val_recs)}  "
          f"Test: {len(test_recs)}")

    train_loader = DataLoader(
        AmyloidDataset(train_recs, DATA_DIR, augment=True),
        batch_size=cfg["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        AmyloidDataset(val_recs, DATA_DIR, augment=False),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        AmyloidDataset(test_recs, DATA_DIR, augment=False),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )

    model = Simple3DCNN(
        channels    = tuple(cfg["channels"]),
        fc_units    = cfg["fc_units"],
        dropout     = cfg["dropout"],
        kernel_size = cfg["kernel_size"],
    ).to(DEVICE)

    criterion = FocalLoss(gamma=cfg["gamma"])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=CV_EPOCHS)

    best_val_f1     = -1.0
    best_threshold  = 0.5
    best_val_labels = None
    best_val_probs  = None
    best_epoch      = 0
    patience_count  = 0
    best_state      = None
    train_losses, val_losses = [], []

    fold_start_time = time.time()

    for epoch in range(CV_EPOCHS):
        ep_start         = time.time()
        tr_loss          = train_one_epoch(model, train_loader,
                                           optimizer, criterion, DEVICE)
        val_loss, vl, vp = eval_pass(model, val_loader,
                                     criterion, DEVICE)
        scheduler.step()
        ep_time = time.time() - ep_start

        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        t       = tune_threshold(vl, vp)
        val_f1  = f1_score(vl, (vp >= t).astype(int),
                           average="macro", zero_division=0)
        val_auc = roc_auc_score(vl, vp)

        is_best = val_f1 > best_val_f1
        print(f"    ep {epoch+1:02d}/{CV_EPOCHS} | "
              f"train={tr_loss:.4f}  val={val_loss:.4f} | "
              f"f1={val_f1:.4f}  auc={val_auc:.4f}  "
              f"thresh={t:.2f}  pat={patience_count}/{CV_PATIENCE}  "
              f"[{ep_time:.1f}s]"
              + (" ← best" if is_best else ""))

        # after epoch 1: print fold ETA
        if epoch == 0:
            remaining_eps = CV_EPOCHS - 1
            print(f"    → ~{ep_time:.1f}s/epoch  |  "
                  f"fold ETA: ~{ep_time * remaining_eps / 60:.1f} min")

        if is_best:
            best_val_f1     = val_f1
            best_threshold  = t
            best_val_labels = vl
            best_val_probs  = vp
            best_epoch      = epoch + 1
            patience_count  = 0
            best_state      = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= CV_PATIENCE:
                print(f"    Early stopping at epoch {epoch+1}")
                break

    # ── Restore best weights & evaluate on test ──
    model.load_state_dict(best_state)
    model.to(DEVICE)

    _, test_labels, test_probs = eval_pass(model, test_loader,
                                           criterion, DEVICE)
    test_metrics = evaluate(test_labels, test_probs, best_threshold)
    val_metrics  = evaluate(best_val_labels, best_val_probs, best_threshold)

    # ── Save fold checkpoint ──────────────────────
    ckpt_path = DIR_CKPT / f"fold_{fold_idx}.pt"
    torch.save({
        "fold"          : fold_idx,
        "state_dict"    : best_state,
        "best_epoch"    : best_epoch,
        "best_threshold": best_threshold,
        "val_metrics"   : val_metrics,
        "test_metrics"  : test_metrics,
        "config"        : cfg,
    }, ckpt_path)

    # ── Save fold loss curve ──────────────────────
    plot_loss_curve(
        train_losses, val_losses,
        title=f"Fold {fold_idx} — Train vs Val Loss",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_loss_curve.png"
    )

    # ── Save fold ROC curve ───────────────────────
    plot_roc(
        test_labels, test_probs,
        title=f"Fold {fold_idx} — ROC Curve (Test)",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_roc.png"
    )

    # ── Save fold confusion matrix ────────────────
    plot_confusion(
        test_metrics["confusion"],
        title=f"Fold {fold_idx} — Confusion Matrix (Test)",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_confusion.png"
    )

    # ── Print fold test summary ───────────────────
    print(f"\n  ── Fold {fold_idx} Test Results ─────────────────")
    print(f"     best_epoch : {best_epoch}")
    print(f"     threshold  : {best_threshold:.2f}")
    print(f"     macro_f1   : {test_metrics['macro_f1']:.4f}")
    print(f"     auc        : {test_metrics['auc']:.4f}")
    print(f"     f1_neg     : {test_metrics['f1_neg']:.4f}  "
          f"(prec={test_metrics['precision_neg']:.4f}  "
          f"rec={test_metrics['recall_neg']:.4f})")
    print(f"     f1_pos     : {test_metrics['f1_pos']:.4f}  "
          f"(prec={test_metrics['precision_pos']:.4f}  "
          f"rec={test_metrics['recall_pos']:.4f})")
    print(f"     sensitivity: {test_metrics['sensitivity']:.4f}")
    print(f"     specificity: {test_metrics['specificity']:.4f}")

    return {
        "fold"          : fold_idx,
        "best_epoch"    : best_epoch,
        "best_threshold": best_threshold,
        "val_metrics"   : val_metrics,
        "test_metrics"  : test_metrics,
        "train_losses"  : train_losses,
        "val_losses"    : val_losses,
        "test_labels"   : test_labels.tolist(),
        "test_probs"    : test_probs.tolist(),
    }

In [14]:
# ── Run all folds ─────────────────────────────
df = pd.read_csv(CSV_PATH)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

cv_results = []
cv_start   = time.time()

print(f"\n{'='*60}")
print(f"  Full CV Training — {N_FOLDS} folds")
print(f"  Config : {CFG['channels']}  lr={CFG['lr']}  "
      f"bs={CFG['batch_size']}  gamma={CFG['gamma']}")
print(f"  Epochs : max={CV_EPOCHS}  patience={CV_PATIENCE}")
print(f"{'='*60}")


  Full CV Training — 7 folds
  Config : [32, 64, 128, 256]  lr=0.001  bs=4  gamma=2.0
  Epochs : max=50  patience=15


In [15]:
for fold_idx in range(N_FOLDS):
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'─'*60}")

    fold_start  = time.time()
    fold_result = train_fold(fold_idx, df, splits, CFG)
    cv_results.append(fold_result)

    fold_time = time.time() - fold_start
    elapsed   = time.time() - cv_start
    remaining = (elapsed / (fold_idx + 1)) * (N_FOLDS - fold_idx - 1)

    print(f"\n  Fold {fold_idx+1} done in    : {fold_time/60:.1f} min")
    print(f"  Total elapsed         : {elapsed/3600:.2f}h")
    print(f"  Folds done            : {fold_idx+1}/{N_FOLDS}")
    print(f"  ETA remaining         : {remaining/3600:.2f}h  "
          f"(~{remaining/60:.0f} min)")
    print(f"  Checkpoint saved      : {DIR_CKPT}/fold_{fold_idx}.pt ✓")

    # save after every fold
    with open(DIR_RESULTS / "cv_results.json", "w") as f:
        json.dump(cv_results, f, indent=2, default=str)
    print(f"  Results saved         : {DIR_RESULTS}/cv_results.json ✓")


────────────────────────────────────────────────────────────
  FOLD 1/7
────────────────────────────────────────────────────────────

  Train: 1005  Val: 141  Test: 195
    ep 01/50 | train=0.1816  val=0.1602 | f1=0.8122  auc=0.8630  thresh=0.50  pat=0/15  [14.1s] ← best
    → ~14.1s/epoch  |  fold ETA: ~11.5 min
    ep 02/50 | train=0.1545  val=0.1363 | f1=0.8129  auc=0.8858  thresh=0.60  pat=0/15  [14.0s] ← best
    ep 03/50 | train=0.1306  val=0.1514 | f1=0.7899  auc=0.9243  thresh=0.40  pat=0/15  [14.0s]
    ep 04/50 | train=0.1248  val=0.1338 | f1=0.8155  auc=0.9371  thresh=0.60  pat=1/15  [14.0s] ← best
    ep 05/50 | train=0.1315  val=0.1177 | f1=0.8840  auc=0.9363  thresh=0.42  pat=0/15  [14.0s] ← best
    ep 06/50 | train=0.1112  val=0.2276 | f1=0.5236  auc=0.9361  thresh=0.40  pat=0/15  [14.0s]
    ep 07/50 | train=0.1184  val=0.0858 | f1=0.8996  auc=0.9632  thresh=0.54  pat=1/15  [14.0s] ← best
    ep 08/50 | train=0.1080  val=0.3068 | f1=0.4380  auc=0.9316  thresh=0.40  pa

In [16]:
# ── Final message ─────────────────────────────
total_time = time.time() - cv_start
print(f"\n{'='*60}")
print(f"  CV complete!")
print(f"  Total time  : {total_time/3600:.2f}h")
print(f"  Checkpoints : {DIR_CKPT}/")
print(f"  Results     : {DIR_RESULTS}/cv_results.json")
print(f"{'='*60}")


  CV complete!
  Total time  : 1.04h
  Checkpoints : simpleCNN/simpleCNN_checkpoints/
  Results     : simpleCNN/simpleCNN_results/cv_results.json


In [17]:
# ── Per-fold results table ────────────────────
print(f"\n{'='*60}")
print(f"  Per-Fold Test Results Summary")
print(f"{'='*60}")
print(f"  {'Fold':<6} {'macro_f1':<10} {'AUC':<10} {'f1_neg':<10} "
      f"{'f1_pos':<10} {'sens':<8} {'spec':<8} {'epoch':<6} {'thresh'}")
print(f"  {'─'*78}")

for r in cv_results:
    tm = r["test_metrics"]
    print(f"  {r['fold']:<6} {tm['macro_f1']:<10.4f} {tm['auc']:<10.4f} "
          f"{tm['f1_neg']:<10.4f} {tm['f1_pos']:<10.4f} "
          f"{tm['sensitivity']:<8.4f} {tm['specificity']:<8.4f} "
          f"{r['best_epoch']:<6} {r['best_threshold']:.2f}")

print(f"  {'─'*78}")

# mean ± std
metrics_keys = ["macro_f1", "auc", "f1_neg", "f1_pos", "sensitivity", "specificity"]
means = {k: np.mean([r["test_metrics"][k] for r in cv_results]) for k in metrics_keys}
stds  = {k: np.std ([r["test_metrics"][k] for r in cv_results]) for k in metrics_keys}

print(f"  {'mean':<6} {means['macro_f1']:<10.4f} {means['auc']:<10.4f} "
      f"{means['f1_neg']:<10.4f} {means['f1_pos']:<10.4f} "
      f"{means['sensitivity']:<8.4f} {means['specificity']:<8.4f}")
print(f"  {'std':<6} {stds['macro_f1']:<10.4f} {stds['auc']:<10.4f} "
      f"{stds['f1_neg']:<10.4f} {stds['f1_pos']:<10.4f} "
      f"{stds['sensitivity']:<8.4f} {stds['specificity']:<8.4f}")
print(f"{'='*60}")


  Per-Fold Test Results Summary
  Fold   macro_f1   AUC        f1_neg     f1_pos     sens     spec     epoch  thresh
  ──────────────────────────────────────────────────────────────────────────────
  0      0.9435     0.9865     0.9458     0.9412     0.9565   0.9320   28     0.49
  1      0.9106     0.9824     0.9080     0.9133     0.9634   0.8605   48     0.41
  2      0.8992     0.9258     0.9083     0.8901     0.8617   0.9340   11     0.53
  3      0.8801     0.9486     0.9013     0.8589     0.7692   0.9813   30     0.52
  4      0.9289     0.9673     0.9395     0.9182     0.8690   0.9806   10     0.43
  5      0.8954     0.9596     0.8995     0.8912     0.9053   0.8868   22     0.56
  6      0.9375     0.9801     0.9388     0.9362     0.9565   0.9200   27     0.44
  ──────────────────────────────────────────────────────────────────────────────
  mean   0.9136     0.9643     0.9202     0.9070     0.8974   0.9279  
  std    0.0219     0.0201     0.0187     0.0268     0.0654   0.0414

In [18]:
# ─────────────────────────────────────────────
# CELL 8 — Holdout Evaluation with Ensembling
# ─────────────────────────────────────────────

# ── Load holdout set ──────────────────────────
df = pd.read_csv(CSV_PATH)
with open(HOLDOUT_PATH) as f:
    holdout = json.load(f)

holdout_recs    = make_records(df, holdout["holdout_patients"])
holdout_loader  = DataLoader(
    AmyloidDataset(holdout_recs, DATA_DIR, augment=False),
    batch_size=CFG["batch_size"], shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Holdout set: {len(holdout_recs)} images  "
      f"({sum(r['AMYLOID_STATUS'] for r in holdout_recs)} positive  "
      f"/ {sum(1-r['AMYLOID_STATUS'] for r in holdout_recs)} negative)")

# ── Load all fold models & collect probs ──────
print("\nLoading fold checkpoints and running inference...")
all_fold_probs  = []   # (n_folds, n_images)
all_thresholds  = []
holdout_labels  = None

criterion = FocalLoss(gamma=CFG["gamma"])

for fold_idx in range(N_FOLDS):
    ckpt = torch.load(DIR_CKPT / f"fold_{fold_idx}.pt",
                      map_location=DEVICE, weights_only=False)

    model = Simple3DCNN(
        channels    = tuple(ckpt["config"]["channels"]),
        fc_units    = ckpt["config"]["fc_units"],
        dropout     = ckpt["config"]["dropout"],
        kernel_size = ckpt["config"]["kernel_size"],
    ).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])

    _, labels, probs = eval_pass(model, holdout_loader, criterion, DEVICE)

    all_fold_probs.append(probs)
    all_thresholds.append(ckpt["best_threshold"])

    if holdout_labels is None:
        holdout_labels = labels

    print(f"  Fold {fold_idx} loaded  |  "
          f"threshold={ckpt['best_threshold']:.2f}  |  "
          f"individual AUC={roc_auc_score(labels, probs):.4f}")

all_fold_probs = np.array(all_fold_probs)   # (7, n_images)
print(f"\nAll folds loaded. Probs matrix: {all_fold_probs.shape}")


# ─────────────────────────────────────────────
# Ensembling Strategy 1 — Mean Probability
# ─────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 1 — Mean Probability")
print(f"{'─'*60}")

probs_mean = all_fold_probs.mean(axis=0)
thresh_mean = tune_threshold(holdout_labels, probs_mean)
metrics_mean = evaluate(holdout_labels, probs_mean, thresh_mean)

print(f"  threshold  : {thresh_mean:.2f}")
print(f"  macro_f1   : {metrics_mean['macro_f1']:.4f}")
print(f"  auc        : {metrics_mean['auc']:.4f}")
print(f"  f1_neg     : {metrics_mean['f1_neg']:.4f}  "
      f"(prec={metrics_mean['precision_neg']:.4f}  "
      f"rec={metrics_mean['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_mean['f1_pos']:.4f}  "
      f"(prec={metrics_mean['precision_pos']:.4f}  "
      f"rec={metrics_mean['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_mean['sensitivity']:.4f}")
print(f"  specificity: {metrics_mean['specificity']:.4f}")


# ─────────────────────────────────────────────
# Ensembling Strategy 2 — Weighted Mean
#   weight = fold val macro_f1
# ─────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 2 — Weighted Mean (by val macro_f1)")
print(f"{'─'*60}")

fold_weights = np.array([r["val_metrics"]["macro_f1"] for r in cv_results])
fold_weights = fold_weights / fold_weights.sum()   # normalize to sum=1

print(f"  Fold weights: {[f'{w:.3f}' for w in fold_weights]}")

probs_weighted = (all_fold_probs * fold_weights[:, None]).sum(axis=0)
thresh_weighted = tune_threshold(holdout_labels, probs_weighted)
metrics_weighted = evaluate(holdout_labels, probs_weighted, thresh_weighted)

print(f"  threshold  : {thresh_weighted:.2f}")
print(f"  macro_f1   : {metrics_weighted['macro_f1']:.4f}")
print(f"  auc        : {metrics_weighted['auc']:.4f}")
print(f"  f1_neg     : {metrics_weighted['f1_neg']:.4f}  "
      f"(prec={metrics_weighted['precision_neg']:.4f}  "
      f"rec={metrics_weighted['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_weighted['f1_pos']:.4f}  "
      f"(prec={metrics_weighted['precision_pos']:.4f}  "
      f"rec={metrics_weighted['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_weighted['sensitivity']:.4f}")
print(f"  specificity: {metrics_weighted['specificity']:.4f}")


# ─────────────────────────────────────────────
# Ensembling Strategy 3 — Majority Vote
#   each fold votes with its own tuned threshold
# ─────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 3 — Majority Vote (per-fold threshold)")
print(f"{'─'*60}")

# each fold predicts 0/1 with its own threshold
fold_votes = np.array([
    (all_fold_probs[i] >= all_thresholds[i]).astype(int)
    for i in range(N_FOLDS)
])  # (7, n_images)

# majority vote: positive if >50% of folds say positive
votes_sum   = fold_votes.sum(axis=0)          # (n_images,)
preds_vote  = (votes_sum > N_FOLDS / 2).astype(int)

# for AUC use vote fraction as a soft score
probs_vote  = votes_sum / N_FOLDS
metrics_vote = evaluate(holdout_labels, probs_vote, threshold=0.5)

print(f"  threshold  : 0.50 (majority = >3.5/7 folds agree)")
print(f"  macro_f1   : {metrics_vote['macro_f1']:.4f}")
print(f"  auc        : {metrics_vote['auc']:.4f}")
print(f"  f1_neg     : {metrics_vote['f1_neg']:.4f}  "
      f"(prec={metrics_vote['precision_neg']:.4f}  "
      f"rec={metrics_vote['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_vote['f1_pos']:.4f}  "
      f"(prec={metrics_vote['precision_pos']:.4f}  "
      f"rec={metrics_vote['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_vote['sensitivity']:.4f}")
print(f"  specificity: {metrics_vote['specificity']:.4f}")


# ─────────────────────────────────────────────
# Comparison Summary
# ─────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Ensembling Comparison — Holdout Set")
print(f"{'='*60}")
print(f"  {'Strategy':<30} {'macro_f1':<10} {'AUC':<10} "
      f"{'sens':<8} {'spec':<8}")
print(f"  {'─'*60}")

strategies = [
    ("Mean probability",          metrics_mean),
    ("Weighted mean (val F1)",     metrics_weighted),
    ("Majority vote",             metrics_vote),
]
for name, m in strategies:
    print(f"  {name:<30} {m['macro_f1']:<10.4f} {m['auc']:<10.4f} "
          f"{m['sensitivity']:<8.4f} {m['specificity']:<8.4f}")

print(f"{'='*60}")

# ── Pick best ensemble & save plots ───────────
best_name, best_probs, best_metrics, best_thresh = max(
    [
        ("mean",     probs_mean,     metrics_mean,     thresh_mean),
        ("weighted", probs_weighted, metrics_weighted, thresh_weighted),
        ("vote",     probs_vote,     metrics_vote,     0.5),
    ],
    key=lambda x: x[2]["macro_f1"]
)

print(f"\n  Best ensemble: {best_name}  "
      f"(macro_f1={best_metrics['macro_f1']:.4f})")

# plots for best ensemble
plot_roc(
    holdout_labels, best_probs,
    title=f"Holdout ROC — {best_name} ensemble",
    save_path=DIR_RESULTS / "holdout_roc.png"
)
plot_confusion(
    best_metrics["confusion"],
    title=f"Holdout Confusion — {best_name} ensemble",
    save_path=DIR_RESULTS / "holdout_confusion.png"
)

# save full holdout results
holdout_summary = {
    "n_images"          : len(holdout_recs),
    "mean"              : {**metrics_mean,     "threshold": thresh_mean},
    "weighted"          : {**metrics_weighted, "threshold": thresh_weighted},
    "majority_vote"     : {**metrics_vote,     "threshold": 0.5},
    "best_ensemble"     : best_name,
}
with open(DIR_RESULTS / "holdout_results.json", "w") as f:
    json.dump(holdout_summary, f, indent=2, default=str)

print(f"  Saved → {DIR_RESULTS}/holdout_results.json")
print(f"  Plots  → holdout_roc.png, holdout_confusion.png")

Holdout set: 158 images  (79.0 positive  / 79.0 negative)

Loading fold checkpoints and running inference...
  Fold 0 loaded  |  threshold=0.49  |  individual AUC=0.9704
  Fold 1 loaded  |  threshold=0.41  |  individual AUC=0.9788
  Fold 2 loaded  |  threshold=0.53  |  individual AUC=0.9505
  Fold 3 loaded  |  threshold=0.52  |  individual AUC=0.9662
  Fold 4 loaded  |  threshold=0.43  |  individual AUC=0.9603
  Fold 5 loaded  |  threshold=0.56  |  individual AUC=0.9625
  Fold 6 loaded  |  threshold=0.44  |  individual AUC=0.9686

All folds loaded. Probs matrix: (7, 158)

────────────────────────────────────────────────────────────
  Ensemble 1 — Mean Probability
────────────────────────────────────────────────────────────
  threshold  : 0.54
  macro_f1   : 0.9237
  auc        : 0.9726
  f1_neg     : 0.9286  (prec=0.8764  rec=0.9873)
  f1_pos     : 0.9189  (prec=0.9855  rec=0.8608)
  sensitivity: 0.8608
  specificity: 0.9873

────────────────────────────────────────────────────────────

# SimpleCNN + Demo

## Dataset Class Update

In [19]:
# ─────────────────────────────────────────────
# CELL — AmyloidDataset with Demographics
# ─────────────────────────────────────────────

class AmyloidDatasetDemo(Dataset):
    """
    Same as AmyloidDataset but also returns normalized demographics:
        age  : (AGE_AT_SCAN - mean) / std  → scalar
        sex  : 0=Female, 1=Male            → scalar
    Returns (image_tensor, demo_tensor, label)
    """

    # normalization stats (computed from full dataset)
    AGE_MEAN = None
    AGE_STD  = None

    def __init__(self, records: list[dict], data_dir: Path,
                 age_mean: float, age_std: float,
                 augment: bool = False):
        self.records  = records
        self.data_dir = data_dir
        self.augment  = augment
        self.age_mean = age_mean
        self.age_std  = age_std

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        label = int(rec["AMYLOID_STATUS"])

        # ── Image ────────────────────────────
        fpath = self.data_dir / f"{rec['image_id_2_str']}.nii.gz"
        img   = nib.load(str(fpath)).get_fdata(dtype=np.float32)

        if self.augment:
            if random.random() < 0.5:
                img = np.flip(img, axis=0).copy()
            if random.random() < 0.5:
                img = np.flip(img, axis=1).copy()
            jitter = (random.random() * 0.04 - 0.02) * img.mean()
            img    = img + jitter

        img = torch.from_numpy(img).unsqueeze(0)   # (1, 91, 109, 91)

        # ── Demographics ─────────────────────
        age  = (float(rec["AGE_AT_SCAN"]) - self.age_mean) / self.age_std
        sex  = 1.0 if str(rec["SEX"]).strip().upper() in ("M", "MALE", "1") else 0.0
        demo = torch.tensor([age, sex], dtype=torch.float32)   # (2,)

        return img, demo, torch.tensor(label, dtype=torch.long)


def make_records_demo(df: pd.DataFrame, patient_list: list) -> list[dict]:
    """Same as make_records but includes AGE_AT_SCAN and SEX."""
    subset = df[df["PTID"].isin(patient_list)]
    return subset[["image_id_2_str", "AMYLOID_STATUS",
                   "PTID", "AGE_AT_SCAN", "SEX"]].to_dict("records")


def compute_age_stats(df: pd.DataFrame) -> tuple[float, float]:
    """Compute age mean/std from full dataset (not just train fold)."""
    mean = df["AGE_AT_SCAN"].mean()
    std  = df["AGE_AT_SCAN"].std()
    return float(mean), float(std)


# ── Compute & print age stats ─────────────────
df_full   = pd.read_csv(CSV_PATH)
AGE_MEAN, AGE_STD = compute_age_stats(df_full)

print(f"Age mean : {AGE_MEAN:.2f}")
print(f"Age std  : {AGE_STD:.2f}")

# ── Smoke test ────────────────────────────────
with open(SPLITS_PATH) as f:
    splits_tmp = json.load(f)

fold0_recs = make_records_demo(df_full, splits_tmp[0]["train_patients"])
ds_demo    = AmyloidDatasetDemo(fold0_recs, DATA_DIR,
                                AGE_MEAN, AGE_STD, augment=False)
img, demo, lbl = ds_demo[0]

print(f"\nImage shape : {img.shape}")
print(f"Demo tensor : {demo}  (age_norm, sex)")
print(f"Label       : {lbl.item()}")
print(f"Dataset size: {len(ds_demo)}")
print("\nAmyloidDatasetDemo ready ✓")

Age mean : 75.07
Age std  : 7.96

Image shape : torch.Size([1, 91, 109, 91])
Demo tensor : tensor([0.8702, 0.0000])  (age_norm, sex)
Label       : 1
Dataset size: 1005

AmyloidDatasetDemo ready ✓


In [20]:
# ─────────────────────────────────────────────
# CELL — Output Directories (Demo Fusion)
# ─────────────────────────────────────────────

DIR_BEST    = Path("simpleCNN/simpleCNN_demo_best_config")
DIR_CKPT    = Path("simpleCNN/simpleCNN_demo_checkpoints")
DIR_RESULTS = Path("simpleCNN/simpleCNN_demo_results")

for d in [DIR_BEST, DIR_CKPT, DIR_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Best config  → {DIR_BEST}")
print(f"Checkpoints  → {DIR_CKPT}")
print(f"Results      → {DIR_RESULTS}")

Best config  → simpleCNN/simpleCNN_demo_best_config
Checkpoints  → simpleCNN/simpleCNN_demo_checkpoints
Results      → simpleCNN/simpleCNN_demo_results


In [21]:
# ─────────────────────────────────────────────
# CELL — Late Fusion CNN + Demographics
# ─────────────────────────────────────────────

class Simple3DCNN_LateFusion(nn.Module):
    """
    Late fusion: CNN image features + demographics concatenated
    after GAP, before the FC head.

    Architecture:
        Image  → Conv blocks → GAP → (B, C)
        Demo   → Linear(2, demo_hidden) → ReLU → (B, demo_hidden)
        Concat → (B, C + demo_hidden) → Dropout → FC → head
    """

    def __init__(self,
                 channels    : tuple = (32, 64, 128, 256),
                 fc_units    : int   = 512,
                 dropout     : float = 0.3,
                 kernel_size : int   = 3,
                 demo_hidden : int   = 16):
        super().__init__()

        # ── Image encoder (same as Simple3DCNN) ──
        blocks = []
        in_ch  = 1
        for out_ch in channels:
            blocks += [
                nn.Conv3d(in_ch, out_ch, kernel_size=kernel_size,
                          padding=kernel_size // 2, bias=False),
                nn.BatchNorm3d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool3d(kernel_size=2, stride=2),
            ]
            in_ch = out_ch

        self.encoder = nn.Sequential(*blocks)
        self.gap      = nn.AdaptiveAvgPool3d(1)

        # ── Demographics branch ───────────────
        self.demo_branch = nn.Sequential(
            nn.Linear(2, demo_hidden),
            nn.ReLU(),
        )

        # ── Fusion head ───────────────────────
        fused_dim = in_ch + demo_hidden
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(fused_dim, fc_units)
        self.head    = nn.Linear(fc_units, 2)

    def forward(self, img, demo):
        # image path
        x = self.encoder(img)
        x = self.gap(x).flatten(1)       # (B, C)

        # demo path
        d = self.demo_branch(demo)        # (B, demo_hidden)

        # fusion
        x = torch.cat([x, d], dim=1)     # (B, C + demo_hidden)
        x = self.dropout(x)
        x = F.relu(self.fc(x))
        return self.head(x)              # (B, 2)

In [22]:
# ── Smoke test ────────────────────────────────
dummy_img  = torch.zeros(2, 1, 91, 109, 91)
dummy_demo = torch.zeros(2, 2)

model_late = Simple3DCNN_LateFusion()
out        = model_late(dummy_img, dummy_demo)
nparams    = sum(p.numel() for p in model_late.parameters() if p.requires_grad)
print(f"Late fusion output : {out.shape}  params={nparams:,}")
print("Late fusion model ready ✓")

Late fusion output : torch.Size([2, 2])  params=1,303,890
Late fusion model ready ✓


In [23]:
# ─────────────────────────────────────────────
# CELL — Early Fusion CNN + Demographics
# ─────────────────────────────────────────────
class Simple3DCNN_EarlyFusion(nn.Module):
    """
    Early fusion: demographics projected and added to image features
    right after GAP — before any FC transformation.
    The demo projection matches the CNN output dim so they can be
    added (not concatenated) — a cleaner interaction signal.

    Architecture:
        Image  → Conv blocks → GAP → (B, C)
        Demo   → Linear(2, C) → ReLU → (B, C)
        Add    → (B, C)  [element-wise sum]
        → Dropout → FC → head
    """

    def __init__(self,
                 channels    : tuple = (32, 64, 128, 256),
                 fc_units    : int   = 512,
                 dropout     : float = 0.3,
                 kernel_size : int   = 3):
        super().__init__()

        # ── Image encoder ─────────────────────
        blocks = []
        in_ch  = 1
        for out_ch in channels:
            blocks += [
                nn.Conv3d(in_ch, out_ch, kernel_size=kernel_size,
                          padding=kernel_size // 2, bias=False),
                nn.BatchNorm3d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool3d(kernel_size=2, stride=2),
            ]
            in_ch = out_ch

        self.encoder = nn.Sequential(*blocks)
        self.gap      = nn.AdaptiveAvgPool3d(1)

        # ── Demographics projection ───────────
        # project to same dim as CNN output so we can add
        self.demo_proj = nn.Sequential(
            nn.Linear(2, in_ch),
            nn.ReLU(),
        )

        # ── Head ──────────────────────────────
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(in_ch, fc_units)
        self.head    = nn.Linear(fc_units, 2)

    def forward(self, img, demo):
        # image path
        x = self.encoder(img)
        x = self.gap(x).flatten(1)       # (B, C)

        # demo path — project to C dims
        d = self.demo_proj(demo)          # (B, C)

        # early fusion: element-wise addition
        x = x + d                         # (B, C)
        x = self.dropout(x)
        x = F.relu(self.fc(x))
        return self.head(x)              # (B, 2)

In [24]:
# ── Smoke test ────────────────────────────────
dummy_img  = torch.zeros(2, 1, 91, 109, 91)
dummy_demo = torch.zeros(2, 2)

model_early = Simple3DCNN_EarlyFusion()
out         = model_early(dummy_img, dummy_demo)
nparams     = sum(p.numel() for p in model_early.parameters() if p.requires_grad)
print(f"Early fusion output : {out.shape}  params={nparams:,}")
print("Early fusion model ready ✓")

Early fusion output : torch.Size([2, 2])  params=1,296,418
Early fusion model ready ✓


In [34]:
# ─────────────────────────────────────────────
# CELL — Train/Eval Passes (Demographics)
# ─────────────────────────────────────────────

def train_one_epoch_demo(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for imgs, demos, labels in loader:
        imgs, demos, labels = (imgs.to(device),
                               demos.to(device),
                               labels.to(device))
        optimizer.zero_grad()
        loss = criterion(model(imgs, demos), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        n          += len(labels)
    return total_loss / n


@torch.no_grad()
def eval_pass_demo(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_labels, all_probs = [], []
    for imgs, demos, labels in loader:
        imgs, demos, labels = (imgs.to(device),
                               demos.to(device),
                               labels.to(device))
        logits = model(imgs, demos)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        n          += len(labels)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs)
    return total_loss / n, np.concatenate(all_labels), np.concatenate(all_probs)


print("Demo train/eval functions ready ✓")

Demo train/eval functions ready ✓


In [35]:
# ─────────────────────────────────────────────
# CELL — Random Search (Late + Early Fusion)
# ─────────────────────────────────────────────
import time

# ── Fixed backbone config (Trial 3) ──────────
BACKBONE_CFG = {
    "channels"    : (32, 64, 128, 256),
    "kernel_size" : 3,
}

# ── Search space (head + training only) ───────
SEARCH_SPACE_DEMO = {
    "fc_units"   : [128, 256, 512],
    "dropout"    : [0.3, 0.4, 0.5, 0.6],
    "lr"         : [1e-5, 5e-5, 1e-4],   # was [1e-4, 3e-4, 5e-4, 1e-3]
    "batch_size" : [4, 8, 16],
    "gamma"      : [0.5, 1.0, 2.0, 3.0],
    "fusion"     : ["late", "early"],
}

SEARCH_EPOCHS_MAX = 25
PATIENCE          = 10
N_TRIALS_DEMO     = 20


def run_trial_demo(config: dict, df: pd.DataFrame,
                   splits: list, fold_idx: int,
                   trial_idx: int) -> dict:

    print(f"    Fusion: {config['fusion'].upper()}")
    fold       = splits[fold_idx]
    train_recs = make_records_demo(df, fold["train_patients"])
    val_recs   = make_records_demo(df, fold["val_patients"])
    print(f"    Train: {len(train_recs)}  |  Val: {len(val_recs)}")

    train_loader = DataLoader(
        AmyloidDatasetDemo(train_recs, DATA_DIR,
                           AGE_MEAN, AGE_STD, augment=True),
        batch_size=config["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        AmyloidDatasetDemo(val_recs, DATA_DIR,
                           AGE_MEAN, AGE_STD, augment=False),
        batch_size=config["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )

    if config["fusion"] == "late":
        model = Simple3DCNN_LateFusion(
            channels    = BACKBONE_CFG["channels"],
            fc_units    = config["fc_units"],
            dropout     = config["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)
    else:
        model = Simple3DCNN_EarlyFusion(
            channels    = BACKBONE_CFG["channels"],
            fc_units    = config["fc_units"],
            dropout     = config["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)

    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Model: fc={config['fc_units']}  drop={config['dropout']}  "
          f"params={nparams:,}")
    print(f"    Optim: lr={config['lr']}  bs={config['batch_size']}  "
          f"gamma={config['gamma']}")

    criterion = FocalLoss(gamma=config["gamma"])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=SEARCH_EPOCHS_MAX)

    best_val_f1     = -1.0
    best_threshold  = 0.5
    best_val_labels = None
    best_val_probs  = None
    patience_count  = 0
    train_losses, val_losses = [], []

    print(f"    Training...")
    for epoch in range(SEARCH_EPOCHS_MAX):
        ep_start         = time.time()
        tr_loss          = train_one_epoch_demo(model, train_loader,
                                                optimizer, criterion, DEVICE)
        val_loss, vl, vp = eval_pass_demo(model, val_loader,
                                          criterion, DEVICE)
        scheduler.step()
        ep_time = time.time() - ep_start

        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        t       = tune_threshold(vl, vp)
        val_f1  = f1_score(vl, (vp >= t).astype(int),
                           average="macro", zero_division=0)
        val_auc = roc_auc_score(vl, vp)

        print(f"      ep {epoch+1:02d}/{SEARCH_EPOCHS_MAX} | "
              f"train={tr_loss:.4f}  val={val_loss:.4f} | "
              f"f1={val_f1:.4f}  auc={val_auc:.4f}  "
              f"thresh={t:.2f}  pat={patience_count}/{PATIENCE}  "
              f"[{ep_time:.1f}s]"
              + (" ← best" if val_f1 > best_val_f1 else ""))

        if epoch == 0:
            print(f"      → ~{ep_time:.1f}s/epoch  |  "
                  f"trial ETA: ~{ep_time * SEARCH_EPOCHS_MAX / 60:.1f} min")

        if val_f1 > best_val_f1:
            best_val_f1     = val_f1
            best_threshold  = t
            best_val_labels = vl
            best_val_probs  = vp
            patience_count  = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"      Early stopping at epoch {epoch+1}")
                break

    metrics = evaluate(best_val_labels, best_val_probs, best_threshold)

    print(f"    ── Trial {trial_idx+1} result ──────────────────────")
    print(f"       fusion       : {config['fusion']}")
    print(f"       val_macro_f1 : {metrics['macro_f1']:.4f}")
    print(f"       val_auc      : {metrics['auc']:.4f}")
    print(f"       f1_neg       : {metrics['f1_neg']:.4f}  "
          f"(prec={metrics['precision_neg']:.4f}  "
          f"rec={metrics['recall_neg']:.4f})")
    print(f"       f1_pos       : {metrics['f1_pos']:.4f}  "
          f"(prec={metrics['precision_pos']:.4f}  "
          f"rec={metrics['recall_pos']:.4f})")
    print(f"       threshold    : {best_threshold:.2f}  |  "
          f"epochs_run: {len(train_losses)}")

    return {
        "trial"            : trial_idx,
        "config"           : config,
        "val_macro_f1"     : metrics["macro_f1"],
        "val_auc"          : metrics["auc"],
        "val_f1_neg"       : metrics["f1_neg"],
        "val_f1_pos"       : metrics["f1_pos"],
        "val_precision_neg": metrics["precision_neg"],
        "val_recall_neg"   : metrics["recall_neg"],
        "val_precision_pos": metrics["precision_pos"],
        "val_recall_pos"   : metrics["recall_pos"],
        "best_threshold"   : best_threshold,
        "epochs_run"       : len(train_losses),
        "train_losses"     : train_losses,
        "val_losses"       : val_losses,
    }


# ── Run the search ────────────────────────────
df = pd.read_csv(CSV_PATH)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

all_results_demo = []
search_start     = time.time()
print(f"{'='*60}")
print(f"  Demo Fusion Random Search — {N_TRIALS_DEMO} trials on fold {SEARCH_FOLD}")
print(f"  Backbone: {BACKBONE_CFG['channels']}")
print(f"  max_epochs={SEARCH_EPOCHS_MAX}  patience={PATIENCE}")
print(f"{'='*60}\n")

for trial in range(N_TRIALS_DEMO):
    torch.cuda.empty_cache()
    config      = sample_config(SEARCH_SPACE_DEMO, seed=SEED + trial)
    trial_start = time.time()

    print(f"{'─'*60}")
    print(f"  TRIAL {trial+1:02d}/{N_TRIALS_DEMO}")
    print(f"{'─'*60}")

    try:
        result = run_trial_demo(config, df, splits, SEARCH_FOLD, trial)
        all_results_demo.append(result)

        best_so_far = max(all_results_demo, key=lambda x: x["val_macro_f1"])
        trial_time  = time.time() - trial_start
        elapsed     = time.time() - search_start
        remaining   = (elapsed / (trial + 1)) * (N_TRIALS_DEMO - trial - 1)

        print(f"    Trial time : {trial_time/60:.1f} min")
        print(f"    Elapsed    : {elapsed/3600:.2f}h  |  "
              f"ETA: {remaining/3600:.2f}h")
        print(f"    Best so far: trial {best_so_far['trial']+1}  "
              f"macro_f1={best_so_far['val_macro_f1']:.4f}  "
              f"fusion={best_so_far['config']['fusion']}")

    except Exception as e:
        print(f"    FAILED — {e}")
        import traceback; traceback.print_exc()

    with open(DIR_BEST / "search_results_demo.json", "w") as f:
        json.dump(all_results_demo, f, indent=2, default=str)
    print(f"    Saved {len(all_results_demo)} trials → "
          f"{DIR_BEST}/search_results_demo.json\n")

# ── Final summary ─────────────────────────────
total_time  = time.time() - search_start
best_result = max(all_results_demo, key=lambda x: x["val_macro_f1"])

print(f"\n{'='*60}")
print(f"  Search complete — {len(all_results_demo)}/{N_TRIALS_DEMO} succeeded")
print(f"  Total time: {total_time/3600:.2f}h")
print(f"{'='*60}")
print(f"  Best trial  : {best_result['trial']+1}")
print(f"  Fusion      : {best_result['config']['fusion']}")
print(f"  Macro F1    : {best_result['val_macro_f1']:.4f}")
print(f"  AUC         : {best_result['val_auc']:.4f}")
print(f"  Config      : {best_result['config']}")
print(f"{'='*60}")

  Demo Fusion Random Search — 20 trials on fold 0
  Backbone: (32, 64, 128, 256)
  max_epochs=25  patience=10

────────────────────────────────────────────────────────────
  TRIAL 01/20
────────────────────────────────────────────────────────────
    Fusion: LATE
    Train: 1005  |  Val: 141
    Model: fc=512  drop=0.3  params=1,303,890
    Optim: lr=1e-05  bs=16  gamma=2.0
    Training...
    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 0 trials → simpleCNN/simpleCNN_demo_best_config/search_results_demo.json

────────────────────────────────────────────────────────────
  TRIAL 02/20
────────────────────────────────────────────────────────────
    Fusion: EARLY
    Train: 1005  |  Val: 141
    Model: fc=128  drop=0.5  params=1,196,962
    Optim: lr=0.0001  bs=4  gamma=3.0
    Training...


Traceback (most recent call last):
  File "/tmp/ipykernel_114187/636624469.py", line 181, in <module>
    result = run_trial_demo(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/636624469.py", line 87, in run_trial_demo
    tr_loss          = train_one_epoch_demo(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1985978997.py", line 14, in train_one_epoch_demo
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    

      ep 01/25 | train=0.0894  val=0.0627 | f1=0.8555  auc=0.8775  thresh=0.52  pat=0/10  [14.1s] ← best
      → ~14.1s/epoch  |  trial ETA: ~5.9 min
      ep 02/25 | train=0.0740  val=0.3183 | f1=0.4679  auc=0.8974  thresh=0.40  pat=0/10  [14.0s]
      ep 03/25 | train=0.0712  val=0.1711 | f1=0.5408  auc=0.9471  thresh=0.60  pat=1/10  [13.9s]
      ep 04/25 | train=0.0686  val=0.2177 | f1=0.7063  auc=0.9253  thresh=0.40  pat=2/10  [13.9s]
      ep 05/25 | train=0.0725  val=0.0755 | f1=0.8430  auc=0.9440  thresh=0.40  pat=3/10  [14.0s]
      ep 06/25 | train=0.0640  val=0.0880 | f1=0.8999  auc=0.9548  thresh=0.43  pat=4/10  [14.0s] ← best
      ep 07/25 | train=0.0650  val=0.2068 | f1=0.7257  auc=0.9265  thresh=0.40  pat=0/10  [14.0s]
      ep 08/25 | train=0.0650  val=0.3586 | f1=0.4219  auc=0.9471  thresh=0.58  pat=1/10  [14.0s]
      ep 09/25 | train=0.0698  val=0.5138 | f1=0.3853  auc=0.9398  thresh=0.57  pat=2/10  [13.9s]
      ep 10/25 | train=0.0646  val=0.1063 | f1=0.9005  auc=

Traceback (most recent call last):
  File "/tmp/ipykernel_114187/636624469.py", line 181, in <module>
    result = run_trial_demo(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/636624469.py", line 87, in run_trial_demo
    tr_loss          = train_one_epoch_demo(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1985978997.py", line 14, in train_one_epoch_demo
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    

    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 3 trials → simpleCNN/simpleCNN_demo_best_config/search_results_demo.json

────────────────────────────────────────────────────────────
  TRIAL 07/20
────────────────────────────────────────────────────────────
    Fusion: LATE
    Train: 1005  |  Val: 141
    Model: fc=512  drop=0.5  params=1,303,890
    Optim: lr=1e-05  bs=16  gamma=2.0
    Training...


...

    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 6 trials → simpleCNN/simpleCNN_demo_best_config/search_results_demo.json

────────────────────────────────────────────────────────────
  TRIAL 17/20
────────────────────────────────────────────────────────────
    Fusion: LATE
    Train: 1005  |  Val: 141
    Model: fc=512  drop=0.4  params=1,303,890
    Optim: lr=1e-05  bs=16  gamma=1.0
    Training...


Traceback (most recent call last):
  File "/tmp/ipykernel_114187/636624469.py", line 181, in <module>
    result = run_trial_demo(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/636624469.py", line 87, in run_trial_demo
    tr_loss          = train_one_epoch_demo(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1985978997.py", line 14, in train_one_epoch_demo
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    

    FAILED — NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1319, please report a bug to PyTorch. 
    Saved 6 trials → simpleCNN/simpleCNN_demo_best_config/search_results_demo.json

────────────────────────────────────────────────────────────
  TRIAL 18/20
────────────────────────────────────────────────────────────
    Fusion: LATE
    Train: 1005  |  Val: 141
    Model: fc=128  drop=0.3  params=1,198,290
    Optim: lr=0.0001  bs=8  gamma=0.5
    Training...


Traceback (most recent call last):
  File "/tmp/ipykernel_114187/636624469.py", line 181, in <module>
    result = run_trial_demo(config, df, splits, SEARCH_FOLD, trial)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/636624469.py", line 87, in run_trial_demo
    tr_loss          = train_one_epoch_demo(model, train_loader,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114187/1985978997.py", line 14, in train_one_epoch_demo
    loss.backward()
  File "/opt/conda/lib/python3.11/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/__init__.py", line 379, in backward
    _engine_run_backward(
  File "/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py", line 882, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    

      ep 01/25 | train=0.4140  val=0.8238 | f1=0.3562  auc=0.9406  thresh=0.40  pat=0/10  [16.0s] ← best
      → ~16.0s/epoch  |  trial ETA: ~6.6 min
      ep 02/25 | train=0.3089  val=0.3869 | f1=0.7418  auc=0.9436  thresh=0.60  pat=0/10  [16.0s] ← best
      ep 03/25 | train=0.2716  val=0.8084 | f1=0.4894  auc=0.9440  thresh=0.58  pat=0/10  [16.0s]
      ep 04/25 | train=0.2901  val=0.2159 | f1=0.9280  auc=0.9689  thresh=0.56  pat=1/10  [16.0s] ← best
      ep 05/25 | train=0.2664  val=0.3602 | f1=0.7899  auc=0.9648  thresh=0.40  pat=0/10  [16.0s]
      ep 06/25 | train=0.2673  val=0.2997 | f1=0.8569  auc=0.9575  thresh=0.40  pat=1/10  [16.0s]
      ep 07/25 | train=0.2821  val=0.3188 | f1=0.8407  auc=0.9424  thresh=0.40  pat=2/10  [15.9s]
      ep 08/25 | train=0.2318  val=0.3243 | f1=0.8324  auc=0.9729  thresh=0.40  pat=3/10  [15.9s]
      ep 09/25 | train=0.2288  val=0.9727 | f1=0.4823  auc=0.9536  thresh=0.40  pat=4/10  [16.0s]
      ep 10/25 | train=0.2482  val=0.1414 | f1=0.949

In [36]:
with open(DIR_BEST / "search_results_demo.json") as f:
    saved_demo = json.load(f)

sorted_demo = sorted(saved_demo, key=lambda x: x["val_macro_f1"], reverse=True)

print(f"  {'Rank':<5} {'Trial':<7} {'Fusion':<8} {'macro_f1':<10} "
      f"{'AUC':<10} {'f1_neg':<10} {'f1_pos':<10} {'epochs':<8} config")
print(f"  {'─'*90}")

for i, r in enumerate(sorted_demo[:5]):
    print(f"  {i+1:<5} {r['trial']+1:<7} {r['config']['fusion']:<8} "
          f"{r['val_macro_f1']:<10.4f} {r['val_auc']:<10.4f} "
          f"{r['val_f1_neg']:<10.4f} {r['val_f1_pos']:<10.4f} "
          f"{r['epochs_run']:<8} "
          f"fc={r['config']['fc_units']} drop={r['config']['dropout']} "
          f"lr={r['config']['lr']} gamma={r['config']['gamma']}")

# best per fusion type
late_results  = [r for r in saved_demo if r["config"]["fusion"] == "late"]
early_results = [r for r in saved_demo if r["config"]["fusion"] == "early"]
best_late  = max(late_results,  key=lambda x: x["val_macro_f1"])
best_early = max(early_results, key=lambda x: x["val_macro_f1"])

print(f"\n  Best LATE  : trial {best_late['trial']+1}  "
      f"macro_f1={best_late['val_macro_f1']:.4f}  auc={best_late['val_auc']:.4f}")
print(f"  Best EARLY : trial {best_early['trial']+1}  "
      f"macro_f1={best_early['val_macro_f1']:.4f}  auc={best_early['val_auc']:.4f}")

  Rank  Trial   Fusion   macro_f1   AUC        f1_neg     f1_pos     epochs   config
  ──────────────────────────────────────────────────────────────────────────────────────────
  1     18      late     0.9499     0.9756     0.9548     0.9449     20       fc=128 drop=0.3 lr=0.0001 gamma=0.5
  2     20      early    0.9422     0.9723     0.9500     0.9344     25       fc=256 drop=0.4 lr=0.0001 gamma=2.0
  3     2       early    0.9346     0.9721     0.9448     0.9244     25       fc=128 drop=0.5 lr=0.0001 gamma=3.0
  4     19      late     0.9283     0.9528     0.9359     0.9206     19       fc=256 drop=0.5 lr=0.0001 gamma=2.0
  5     8       late     0.9278     0.9689     0.9375     0.9180     25       fc=128 drop=0.5 lr=5e-05 gamma=2.0

  Best LATE  : trial 18  macro_f1=0.9499  auc=0.9756
  Best EARLY : trial 20  macro_f1=0.9422  auc=0.9723


In [37]:
# ── Automatic selection — best macro_f1 within param limit ──
PARAM_LIMIT_DEMO = 2_000_000

candidates_demo = []
for r in saved_demo:
    if r["config"]["fusion"] == "late":
        model_tmp = Simple3DCNN_LateFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = r["config"]["fc_units"],
            dropout     = r["config"]["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        )
    else:
        model_tmp = Simple3DCNN_EarlyFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = r["config"]["fc_units"],
            dropout     = r["config"]["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        )
    nparams = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
    if nparams <= PARAM_LIMIT_DEMO:
        candidates_demo.append((r, nparams))

candidates_demo.sort(key=lambda x: x[0]["val_macro_f1"], reverse=True)
chosen_demo, chosen_demo_params = candidates_demo[0]
CHOSEN_TRIAL_DEMO = chosen_demo["trial"] + 1

print(f"  Param limit   : {PARAM_LIMIT_DEMO:,}")
print(f"  Candidates    : {len(candidates_demo)}/{len(saved_demo)} trials under limit")
print(f"  Auto-selected : Trial {CHOSEN_TRIAL_DEMO}  "
      f"macro_f1={chosen_demo['val_macro_f1']:.4f}  "
      f"fusion={chosen_demo['config']['fusion']}  "
      f"params={chosen_demo_params:,}")

config_path = DIR_BEST / "best_config_demo.json"
with open(config_path, "w") as f:
    json.dump(chosen_demo, f, indent=2, default=str)

print(f"\n{'='*60}")
print(f"  Selected — Trial {CHOSEN_TRIAL_DEMO}  "
      f"({chosen_demo['config']['fusion'].upper()} fusion)")
print(f"{'='*60}")
print(f"  Macro F1   : {chosen_demo['val_macro_f1']:.4f}")
print(f"  AUC        : {chosen_demo['val_auc']:.4f}")
print(f"  f1_neg     : {chosen_demo['val_f1_neg']:.4f}  "
      f"(prec={chosen_demo['val_precision_neg']:.4f}  "
      f"rec={chosen_demo['val_recall_neg']:.4f})")
print(f"  f1_pos     : {chosen_demo['val_f1_pos']:.4f}  "
      f"(prec={chosen_demo['val_precision_pos']:.4f}  "
      f"rec={chosen_demo['val_recall_pos']:.4f})")
print(f"  Threshold  : {chosen_demo['best_threshold']:.2f}")
print(f"  Epochs run : {chosen_demo['epochs_run']}")
print(f"{'─'*60}")
for k, v in chosen_demo['config'].items():
    print(f"  {k:12s}: {v}")
print(f"{'='*60}")
print(f"  Saved → {config_path}")

plot_loss_curve(
    chosen_demo["train_losses"],
    chosen_demo["val_losses"],
    title=f"Search — Trial {CHOSEN_TRIAL_DEMO} "
          f"({chosen_demo['config']['fusion']} fusion) Loss Curve",
    save_path=DIR_BEST / "chosen_trial_loss_curve_demo.png"
)
print(f"  Loss curve → {DIR_BEST}/chosen_trial_loss_curve_demo.png")

  Param limit   : 2,000,000
  Candidates    : 9/9 trials under limit
  Auto-selected : Trial 18  macro_f1=0.9499  fusion=late  params=1,198,290

  Selected — Trial 18  (LATE fusion)
  Macro F1   : 0.9499
  AUC        : 0.9756
  f1_neg     : 0.9548  (prec=0.9610  rec=0.9487)
  f1_pos     : 0.9449  (prec=0.9375  rec=0.9524)
  Threshold  : 0.45
  Epochs run : 20
────────────────────────────────────────────────────────────
  fc_units    : 128
  dropout     : 0.3
  lr          : 0.0001
  batch_size  : 8
  gamma       : 0.5
  fusion      : late
  Saved → simpleCNN/simpleCNN_demo_best_config/best_config_demo.json
  Loss curve → simpleCNN/simpleCNN_demo_best_config/chosen_trial_loss_curve_demo.png


In [38]:
# ─────────────────────────────────────────────
# CELL — train_fold_demo function
# ─────────────────────────────────────────────

def train_fold_demo(fold_idx: int, df: pd.DataFrame,
                    splits: list, cfg: dict) -> dict:

    fold       = splits[fold_idx]
    train_recs = make_records_demo(df, fold["train_patients"])
    val_recs   = make_records_demo(df, fold["val_patients"])
    test_recs  = make_records_demo(df, fold["test_patients"])

    print(f"\n  Train: {len(train_recs)}  Val: {len(val_recs)}  "
          f"Test: {len(test_recs)}")

    train_loader = DataLoader(
        AmyloidDatasetDemo(train_recs, DATA_DIR,
                           AGE_MEAN, AGE_STD, augment=True),
        batch_size=cfg["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        AmyloidDatasetDemo(val_recs, DATA_DIR,
                           AGE_MEAN, AGE_STD, augment=False),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        AmyloidDatasetDemo(test_recs, DATA_DIR,
                           AGE_MEAN, AGE_STD, augment=False),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=2, pin_memory=True
    )

    if cfg["fusion"] == "late":
        model = Simple3DCNN_LateFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = cfg["fc_units"],
            dropout     = cfg["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)
    else:
        model = Simple3DCNN_EarlyFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = cfg["fc_units"],
            dropout     = cfg["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)

    criterion = FocalLoss(gamma=cfg["gamma"])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=CV_EPOCHS)

    best_val_f1     = -1.0
    best_threshold  = 0.5
    best_val_labels = None
    best_val_probs  = None
    best_epoch      = 0
    patience_count  = 0
    best_state      = None
    train_losses, val_losses = [], []

    for epoch in range(CV_EPOCHS):
        ep_start         = time.time()
        tr_loss          = train_one_epoch_demo(model, train_loader,
                                                optimizer, criterion, DEVICE)
        val_loss, vl, vp = eval_pass_demo(model, val_loader,
                                          criterion, DEVICE)
        scheduler.step()
        ep_time = time.time() - ep_start

        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        t       = tune_threshold(vl, vp)
        val_f1  = f1_score(vl, (vp >= t).astype(int),
                           average="macro", zero_division=0)
        val_auc = roc_auc_score(vl, vp)

        is_best = val_f1 > best_val_f1
        print(f"    ep {epoch+1:02d}/{CV_EPOCHS} | "
              f"train={tr_loss:.4f}  val={val_loss:.4f} | "
              f"f1={val_f1:.4f}  auc={val_auc:.4f}  "
              f"thresh={t:.2f}  pat={patience_count}/{CV_PATIENCE}  "
              f"[{ep_time:.1f}s]"
              + (" ← best" if is_best else ""))

        if epoch == 0:
            print(f"    → ~{ep_time:.1f}s/epoch  |  "
                  f"fold ETA: ~{ep_time * CV_EPOCHS / 60:.1f} min")

        if is_best:
            best_val_f1     = val_f1
            best_threshold  = t
            best_val_labels = vl
            best_val_probs  = vp
            best_epoch      = epoch + 1
            patience_count  = 0
            best_state      = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= CV_PATIENCE:
                print(f"    Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    _, test_labels, test_probs = eval_pass_demo(model, test_loader,
                                                criterion, DEVICE)
    test_metrics = evaluate(test_labels, test_probs, best_threshold)
    val_metrics  = evaluate(best_val_labels, best_val_probs, best_threshold)

    ckpt_path = DIR_CKPT / f"fold_{fold_idx}.pt"
    torch.save({
        "fold"          : fold_idx,
        "state_dict"    : best_state,
        "best_epoch"    : best_epoch,
        "best_threshold": best_threshold,
        "val_metrics"   : val_metrics,
        "test_metrics"  : test_metrics,
        "config"        : cfg,
        "fusion"        : cfg["fusion"],
    }, ckpt_path)

    plot_loss_curve(
        train_losses, val_losses,
        title=f"Fold {fold_idx} ({cfg['fusion']} fusion) — Train vs Val Loss",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_loss_curve.png"
    )
    plot_roc(
        test_labels, test_probs,
        title=f"Fold {fold_idx} — ROC Curve (Test)",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_roc.png"
    )
    plot_confusion(
        test_metrics["confusion"],
        title=f"Fold {fold_idx} — Confusion Matrix (Test)",
        save_path=DIR_RESULTS / f"fold_{fold_idx}_confusion.png"
    )

    print(f"\n  ── Fold {fold_idx} Test Results ─────────────────")
    print(f"     fusion     : {cfg['fusion']}")
    print(f"     best_epoch : {best_epoch}")
    print(f"     threshold  : {best_threshold:.2f}")
    print(f"     macro_f1   : {test_metrics['macro_f1']:.4f}")
    print(f"     auc        : {test_metrics['auc']:.4f}")
    print(f"     f1_neg     : {test_metrics['f1_neg']:.4f}  "
          f"(prec={test_metrics['precision_neg']:.4f}  "
          f"rec={test_metrics['recall_neg']:.4f})")
    print(f"     f1_pos     : {test_metrics['f1_pos']:.4f}  "
          f"(prec={test_metrics['precision_pos']:.4f}  "
          f"rec={test_metrics['recall_pos']:.4f})")
    print(f"     sensitivity: {test_metrics['sensitivity']:.4f}")
    print(f"     specificity: {test_metrics['specificity']:.4f}")

    return {
        "fold"          : fold_idx,
        "best_epoch"    : best_epoch,
        "best_threshold": best_threshold,
        "val_metrics"   : val_metrics,
        "test_metrics"  : test_metrics,
        "train_losses"  : train_losses,
        "val_losses"    : val_losses,
        "test_labels"   : test_labels.tolist(),
        "test_probs"    : test_probs.tolist(),
    }


print("train_fold_demo ready ✓")

train_fold_demo ready ✓


In [39]:
# ─────────────────────────────────────────────
# CELL — Full 7-Fold CV Training (Demo Fusion)
# ─────────────────────────────────────────────

with open(DIR_BEST / "best_config_demo.json") as f:
    best_demo = json.load(f)

CFG_DEMO = best_demo["config"]
# ── Override unstable hyperparameters ────────
CFG_DEMO["lr"]         = 1e-4    # was 5e-4 — too aggressive
CFG_DEMO["batch_size"] = 8       # was 16 — reduces gradient noise

print("Config after override:")
for k, v in CFG_DEMO.items():
    print(f"  {k:12s}: {v}")

CV_EPOCHS   = 50
CV_PATIENCE = 15

df = pd.read_csv(CSV_PATH)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

cv_results_demo = []
cv_start        = time.time()

print(f"\n{'='*60}")
print(f"  Full CV — Demo Fusion ({CFG_DEMO['fusion'].upper()}) — {N_FOLDS} folds")
print(f"  Backbone : {BACKBONE_CFG['channels']}")
print(f"  Config   : fc={CFG_DEMO['fc_units']}  lr={CFG_DEMO['lr']}  "
      f"bs={CFG_DEMO['batch_size']}  gamma={CFG_DEMO['gamma']}")
print(f"  Epochs   : max={CV_EPOCHS}  patience={CV_PATIENCE}")
print(f"{'='*60}")

for fold_idx in range(N_FOLDS):
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'─'*60}")

    fold_start  = time.time()
    fold_result = train_fold_demo(fold_idx, df, splits, CFG_DEMO)
    cv_results_demo.append(fold_result)

    fold_time = time.time() - fold_start
    elapsed   = time.time() - cv_start
    remaining = (elapsed / (fold_idx + 1)) * (N_FOLDS - fold_idx - 1)

    print(f"\n  Fold {fold_idx+1} done in    : {fold_time/60:.1f} min")
    print(f"  Total elapsed         : {elapsed/3600:.2f}h")
    print(f"  Folds done            : {fold_idx+1}/{N_FOLDS}")
    print(f"  ETA remaining         : {remaining/3600:.2f}h")
    print(f"  Checkpoint saved      : {DIR_CKPT}/fold_{fold_idx}.pt ✓")

    with open(DIR_RESULTS / "cv_results_demo.json", "w") as f:
        json.dump(cv_results_demo, f, indent=2, default=str)
    print(f"  Results saved         : {DIR_RESULTS}/cv_results_demo.json ✓")

# ── Per-fold summary table ────────────────────
total_time = time.time() - cv_start
print(f"\n{'='*60}")
print(f"  Per-Fold Test Results — {CFG_DEMO['fusion'].upper()} Fusion")
print(f"{'='*60}")
print(f"  {'Fold':<6} {'macro_f1':<10} {'AUC':<10} {'f1_neg':<10} "
      f"{'f1_pos':<10} {'sens':<8} {'spec':<8} {'epoch':<6} {'thresh'}")
print(f"  {'─'*78}")

for r in cv_results_demo:
    tm = r["test_metrics"]
    print(f"  {r['fold']:<6} {tm['macro_f1']:<10.4f} {tm['auc']:<10.4f} "
          f"{tm['f1_neg']:<10.4f} {tm['f1_pos']:<10.4f} "
          f"{tm['sensitivity']:<8.4f} {tm['specificity']:<8.4f} "
          f"{r['best_epoch']:<6} {r['best_threshold']:.2f}")

print(f"  {'─'*78}")
metrics_keys = ["macro_f1", "auc", "f1_neg", "f1_pos", "sensitivity", "specificity"]
means = {k: np.mean([r["test_metrics"][k] for r in cv_results_demo])
         for k in metrics_keys}
stds  = {k: np.std ([r["test_metrics"][k] for r in cv_results_demo])
         for k in metrics_keys}

print(f"  {'mean':<6} {means['macro_f1']:<10.4f} {means['auc']:<10.4f} "
      f"{means['f1_neg']:<10.4f} {means['f1_pos']:<10.4f} "
      f"{means['sensitivity']:<8.4f} {means['specificity']:<8.4f}")
print(f"  {'std':<6} {stds['macro_f1']:<10.4f} {stds['auc']:<10.4f} "
      f"{stds['f1_neg']:<10.4f} {stds['f1_pos']:<10.4f} "
      f"{stds['sensitivity']:<8.4f} {stds['specificity']:<8.4f}")
print(f"\n  Total time: {total_time/3600:.2f}h")
print(f"{'='*60}")

Config after override:
  fc_units    : 128
  dropout     : 0.3
  lr          : 0.0001
  batch_size  : 8
  gamma       : 0.5
  fusion      : late

  Full CV — Demo Fusion (LATE) — 7 folds
  Backbone : (32, 64, 128, 256)
  Config   : fc=128  lr=0.0001  bs=8  gamma=0.5
  Epochs   : max=50  patience=15

────────────────────────────────────────────────────────────
  FOLD 1/7
────────────────────────────────────────────────────────────

  Train: 1005  Val: 141  Test: 195
    ep 01/50 | train=0.4440  val=0.5602 | f1=0.5295  auc=0.8940  thresh=0.60  pat=0/15  [16.0s] ← best
    → ~16.0s/epoch  |  fold ETA: ~13.4 min
    ep 02/50 | train=0.3176  val=1.4828 | f1=0.3562  auc=0.9066  thresh=0.40  pat=0/15  [16.0s]
    ep 03/50 | train=0.2891  val=0.6582 | f1=0.6759  auc=0.8940  thresh=0.40  pat=1/15  [16.1s] ← best
    ep 04/50 | train=0.2715  val=0.8125 | f1=0.5236  auc=0.9477  thresh=0.40  pat=0/15  [16.1s]
    ep 05/50 | train=0.2972  val=0.3208 | f1=0.8337  auc=0.9577  thresh=0.40  pat=1/15  [

In [40]:
# ─────────────────────────────────────────────
# CELL — Holdout Evaluation — Demo Fusion
# ─────────────────────────────────────────────

DIR_CKPT    = Path("simpleCNN/simpleCNN_demo_checkpoints")
DIR_RESULTS = Path("simpleCNN/simpleCNN_demo_results")

# ── Load holdout set ──────────────────────────
df = pd.read_csv(CSV_PATH)
with open(HOLDOUT_PATH) as f:
    holdout = json.load(f)

holdout_recs   = make_records_demo(df, holdout["holdout_patients"])
holdout_loader = DataLoader(
    AmyloidDatasetDemo(holdout_recs, DATA_DIR,
                       AGE_MEAN, AGE_STD, augment=False),
    batch_size=CFG_DEMO["batch_size"], shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Holdout set: {len(holdout_recs)} images  "
      f"({sum(r['AMYLOID_STATUS'] for r in holdout_recs)} positive  "
      f"/ {sum(1-r['AMYLOID_STATUS'] for r in holdout_recs)} negative)")

# ── Load all fold models & collect probs ──────
print("\nLoading fold checkpoints and running inference...")
all_fold_probs = []
all_thresholds = []
holdout_labels = None

criterion = FocalLoss(gamma=CFG_DEMO["gamma"])

for fold_idx in range(N_FOLDS):
    ckpt = torch.load(DIR_CKPT / f"fold_{fold_idx}.pt",
                      map_location=DEVICE, weights_only=False)

    if ckpt["fusion"] == "late":
        model = Simple3DCNN_LateFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = ckpt["config"]["fc_units"],
            dropout     = ckpt["config"]["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)
    else:
        model = Simple3DCNN_EarlyFusion(
            channels    = tuple(BACKBONE_CFG["channels"]),
            fc_units    = ckpt["config"]["fc_units"],
            dropout     = ckpt["config"]["dropout"],
            kernel_size = BACKBONE_CFG["kernel_size"],
        ).to(DEVICE)

    model.load_state_dict(ckpt["state_dict"])

    _, labels, probs = eval_pass_demo(model, holdout_loader, criterion, DEVICE)

    all_fold_probs.append(probs)
    all_thresholds.append(ckpt["best_threshold"])

    if holdout_labels is None:
        holdout_labels = labels

    print(f"  Fold {fold_idx} loaded  |  "
          f"threshold={ckpt['best_threshold']:.2f}  |  "
          f"individual AUC={roc_auc_score(labels, probs):.4f}")

all_fold_probs = np.array(all_fold_probs)
print(f"\nAll folds loaded. Probs matrix: {all_fold_probs.shape}")

# ── Ensemble 1 — Mean Probability ─────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 1 — Mean Probability")
print(f"{'─'*60}")
probs_mean    = all_fold_probs.mean(axis=0)
thresh_mean   = tune_threshold(holdout_labels, probs_mean)
metrics_mean  = evaluate(holdout_labels, probs_mean, thresh_mean)
print(f"  threshold  : {thresh_mean:.2f}")
print(f"  macro_f1   : {metrics_mean['macro_f1']:.4f}")
print(f"  auc        : {metrics_mean['auc']:.4f}")
print(f"  f1_neg     : {metrics_mean['f1_neg']:.4f}  "
      f"(prec={metrics_mean['precision_neg']:.4f}  "
      f"rec={metrics_mean['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_mean['f1_pos']:.4f}  "
      f"(prec={metrics_mean['precision_pos']:.4f}  "
      f"rec={metrics_mean['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_mean['sensitivity']:.4f}")
print(f"  specificity: {metrics_mean['specificity']:.4f}")

# ── Ensemble 2 — Weighted Mean ─────────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 2 — Weighted Mean (by val macro_f1)")
print(f"{'─'*60}")
fold_weights     = np.array([r["val_metrics"]["macro_f1"]
                             for r in cv_results_demo])
fold_weights     = fold_weights / fold_weights.sum()
probs_weighted   = (all_fold_probs * fold_weights[:, None]).sum(axis=0)
thresh_weighted  = tune_threshold(holdout_labels, probs_weighted)
metrics_weighted = evaluate(holdout_labels, probs_weighted, thresh_weighted)
print(f"  Fold weights: {[f'{w:.3f}' for w in fold_weights]}")
print(f"  threshold  : {thresh_weighted:.2f}")
print(f"  macro_f1   : {metrics_weighted['macro_f1']:.4f}")
print(f"  auc        : {metrics_weighted['auc']:.4f}")
print(f"  f1_neg     : {metrics_weighted['f1_neg']:.4f}  "
      f"(prec={metrics_weighted['precision_neg']:.4f}  "
      f"rec={metrics_weighted['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_weighted['f1_pos']:.4f}  "
      f"(prec={metrics_weighted['precision_pos']:.4f}  "
      f"rec={metrics_weighted['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_weighted['sensitivity']:.4f}")
print(f"  specificity: {metrics_weighted['specificity']:.4f}")

# ── Ensemble 3 — Majority Vote ─────────────────
print(f"\n{'─'*60}")
print(f"  Ensemble 3 — Majority Vote (per-fold threshold)")
print(f"{'─'*60}")
fold_votes   = np.array([
    (all_fold_probs[i] >= all_thresholds[i]).astype(int)
    for i in range(N_FOLDS)
])
votes_sum    = fold_votes.sum(axis=0)
probs_vote   = votes_sum / N_FOLDS
metrics_vote = evaluate(holdout_labels, probs_vote, threshold=0.5)
print(f"  threshold  : 0.50 (majority = >3.5/7 folds agree)")
print(f"  macro_f1   : {metrics_vote['macro_f1']:.4f}")
print(f"  auc        : {metrics_vote['auc']:.4f}")
print(f"  f1_neg     : {metrics_vote['f1_neg']:.4f}  "
      f"(prec={metrics_vote['precision_neg']:.4f}  "
      f"rec={metrics_vote['recall_neg']:.4f})")
print(f"  f1_pos     : {metrics_vote['f1_pos']:.4f}  "
      f"(prec={metrics_vote['precision_pos']:.4f}  "
      f"rec={metrics_vote['recall_pos']:.4f})")
print(f"  sensitivity: {metrics_vote['sensitivity']:.4f}")
print(f"  specificity: {metrics_vote['specificity']:.4f}")

# ── Comparison Summary ────────────────────────
print(f"\n{'='*60}")
print(f"  Ensembling Comparison — Holdout (Early Fusion)")
print(f"{'='*60}")
print(f"  {'Strategy':<30} {'macro_f1':<10} {'AUC':<10} "
      f"{'sens':<8} {'spec':<8}")
print(f"  {'─'*60}")
for name, m in [("Mean probability",      metrics_mean),
                ("Weighted mean (val F1)", metrics_weighted),
                ("Majority vote",          metrics_vote)]:
    print(f"  {name:<30} {m['macro_f1']:<10.4f} {m['auc']:<10.4f} "
          f"{m['sensitivity']:<8.4f} {m['specificity']:<8.4f}")
print(f"{'='*60}")

# ── Best ensemble → save ──────────────────────
best_name, best_probs, best_metrics, best_thresh = max(
    [("mean",     probs_mean,     metrics_mean,     thresh_mean),
     ("weighted", probs_weighted, metrics_weighted, thresh_weighted),
     ("vote",     probs_vote,     metrics_vote,     0.5)],
    key=lambda x: x[2]["macro_f1"]
)
print(f"\n  Best ensemble: {best_name}  "
      f"(macro_f1={best_metrics['macro_f1']:.4f})")

plot_roc(holdout_labels, best_probs,
         title=f"Holdout ROC — {best_name} ensemble (Early Fusion)",
         save_path=DIR_RESULTS / "holdout_roc.png")
plot_confusion(best_metrics["confusion"],
               title=f"Holdout Confusion — {best_name} ensemble (Early Fusion)",
               save_path=DIR_RESULTS / "holdout_confusion.png")

holdout_summary = {
    "n_images"      : len(holdout_recs),
    "fusion"        : CFG_DEMO["fusion"],
    "mean"          : {**metrics_mean,     "threshold": thresh_mean},
    "weighted"      : {**metrics_weighted, "threshold": thresh_weighted},
    "majority_vote" : {**metrics_vote,     "threshold": 0.5},
    "best_ensemble" : best_name,
}
with open(DIR_RESULTS / "holdout_results.json", "w") as f:
    json.dump(holdout_summary, f, indent=2, default=str)
print(f"  Saved → {DIR_RESULTS}/holdout_results.json")
print(f"  Plots  → holdout_roc.png, holdout_confusion.png")

Holdout set: 158 images  (79.0 positive  / 79.0 negative)

Loading fold checkpoints and running inference...
  Fold 0 loaded  |  threshold=0.47  |  individual AUC=0.9814
  Fold 1 loaded  |  threshold=0.40  |  individual AUC=0.9819
  Fold 2 loaded  |  threshold=0.56  |  individual AUC=0.9841
  Fold 3 loaded  |  threshold=0.57  |  individual AUC=0.9801
  Fold 4 loaded  |  threshold=0.40  |  individual AUC=0.9772
  Fold 5 loaded  |  threshold=0.60  |  individual AUC=0.9742
  Fold 6 loaded  |  threshold=0.52  |  individual AUC=0.9768

All folds loaded. Probs matrix: (7, 158)

────────────────────────────────────────────────────────────
  Ensemble 1 — Mean Probability
────────────────────────────────────────────────────────────
  threshold  : 0.53
  macro_f1   : 0.9177
  auc        : 0.9811
  f1_neg     : 0.9172  (prec=0.9231  rec=0.9114)
  f1_pos     : 0.9182  (prec=0.9125  rec=0.9241)
  sensitivity: 0.9241
  specificity: 0.9114

────────────────────────────────────────────────────────────